
<a href="https://colab.research.google.com/github/CienciaDatosUdea/002_EstudiantesAprendizajeEstadistico/blob/main/semestre2026-2/Laboratorios/Laboratorio_07_nn_aplicacionV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Laboratorio 8. Redes neuronales y ruido aerodinámico


En este laboratorio construiremos un modelo que predice el nivel sonoro de un perfil aerodinámico a partir de cinco magnitudes físicas. para estudiar el cambio de la calidad de la predicción cuando evaluamos una velocidad superior a las utilizadas para entrenar

Trabajaremos con Airfoil Self-Noise (NASA/UCI), un conjunto pequeño de 1.503 observaciones.  La red neuronal es un perceptrón multicapa, implementado con `MLPRegressor`.

**Al finalizar podrás:** organizar un problema de regresión, separar los usos de entrenamiento, validación y prueba; interpretar el escalado y las capas de una red; seleccionar una configuración y relacionar sus errores con las condiciones físicas.


## 0. Preparar el entorno
El archivo incluye una copia del conjunto de datos, de modo que la carga funciona también sin conexión. Ejecuta la siguiente celda si necesitas instalar las bibliotecas: cambia `INSTALAR` a `True`. En un entorno que ya las contiene, continúa con los imports.


In [1]:
INSTALAR = False
if INSTALAR:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'numpy', 'pandas', 'matplotlib', 'scikit-learn'])


In [2]:
import copy
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': True})
print('scikit-learn:', sklearn.__version__)
print('Semilla de los experimentos:', SEED)


scikit-learn: 1.2.1
Semilla de los experimentos: 42


## 1. Entender el sistema y la pregunta
Un perfil aerodinámico es la sección de un ala. El flujo de aire alrededor del perfil produce fluctuaciones de presión asociadas al sonido. Las observaciones proceden de ensayos con perfiles NACA 0012 en un túnel de viento anecoico, diseñado para reducir las reflexiones acústicas.

Cada fila representa una medición a una frecuencia y unas condiciones determinadas. Un conjunto de filas con condiciones comunes permite observar un espectro: nivel sonoro en función de la frecuencia.

Las cinco entradas son:

- **Frecuencia $f$ [Hz]:** número de oscilaciones por segundo.
- **Ángulo de ataque $\alpha$ [grados]:** orientación del perfil respecto al flujo.
- **Cuerda $c$ [m]:** distancia característica entre los bordes de ataque y de salida.
- **Velocidad $U$ [m/s]:** velocidad del flujo libre.
- **Espesor de desplazamiento $\delta^*$ [m]:** medida del déficit de caudal producido por la capa límite en el lado de succión.

La salida es el **nivel de presión sonora escalado $L$ [dB]**. Para un nivel convencional, $L_p=20\log_{10}(p_\mathrm{rms}/p_\mathrm{ref})$. Aquí aprenderemos directamente el valor escalado publicado. Un error de 3 dB se interpreta en la escala del objetivo; convertirlo a presión absoluta requiere conocer el escalado.

La tarea de regresión consiste en aprender
$$\hat L=g_\theta(f,\alpha,c,U,\delta^*).$$
Los parámetros $\theta$ se ajustan usando ejemplos. Después evaluamos la función en observaciones reservadas.

**Fuente:** Brooks, T., Pope, D. y Marcolini, M. (1989), [Airfoil Self-Noise](https://doi.org/10.24432/C5VW2C), NASA/UCI, licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Ficha del conjunto](https://archive.ics.uci.edu/dataset/291/airfoil+self+noise). La envergadura y la posición del observador fueron constantes en los ensayos.


### Actividad 1. Formular hipótesis

Escribe dos hipótesis: una sobre la relación entre frecuencia y nivel sonoro, y otra sobre el efecto de la velocidad. Para cada una, describe qué variables mantendrías constantes y qué gráfica observarías.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 2. Cargar y explorar los datos
La siguiente celda contiene el texto original del archivo de UCI. Puedes mantenerla contraída y ejecutarla. Las seis columnas se asignan en el orden de la fuente.


In [ ]:
DATOS_UCI = '800\t0\t0.3048\t71.3\t0.00266337\t126.201\r\n1000\t0\t0.3048\t71.3\t0.00266337\t125.201\r\n1250\t0\t0.3048\t71.3\t0.00266337\t125.951\r\n1600\t0\t0.3048\t71.3\t0.00266337\t127.591\r\n2000\t0\t0.3048\t71.3\t0.00266337\t127.461\r\n2500\t0\t0.3048\t71.3\t0.00266337\t125.571\r\n3150\t0\t0.3048\t71.3\t0.00266337\t125.201\r\n4000\t0\t0.3048\t71.3\t0.00266337\t123.061\r\n5000\t0\t0.3048\t71.3\t0.00266337\t121.301\r\n6300\t0\t0.3048\t71.3\t0.00266337\t119.541\r\n8000\t0\t0.3048\t71.3\t0.00266337\t117.151\r\n10000\t0\t0.3048\t71.3\t0.00266337\t115.391\r\n12500\t0\t0.3048\t71.3\t0.00266337\t112.241\r\n16000\t0\t0.3048\t71.3\t0.00266337\t108.721\r\n500\t0\t0.3048\t55.5\t0.00283081\t126.416\r\n630\t0\t0.3048\t55.5\t0.00283081\t127.696\r\n800\t0\t0.3048\t55.5\t0.00283081\t128.086\r\n1000\t0\t0.3048\t55.5\t0.00283081\t126.966\r\n1250\t0\t0.3048\t55.5\t0.00283081\t126.086\r\n1600\t0\t0.3048\t55.5\t0.00283081\t126.986\r\n2000\t0\t0.3048\t55.5\t0.00283081\t126.616\r\n2500\t0\t0.3048\t55.5\t0.00283081\t124.106\r\n3150\t0\t0.3048\t55.5\t0.00283081\t123.236\r\n4000\t0\t0.3048\t55.5\t0.00283081\t121.106\r\n5000\t0\t0.3048\t55.5\t0.00283081\t119.606\r\n6300\t0\t0.3048\t55.5\t0.00283081\t117.976\r\n8000\t0\t0.3048\t55.5\t0.00283081\t116.476\r\n10000\t0\t0.3048\t55.5\t0.00283081\t113.076\r\n12500\t0\t0.3048\t55.5\t0.00283081\t111.076\r\n200\t0\t0.3048\t39.6\t0.00310138\t118.129\r\n250\t0\t0.3048\t39.6\t0.00310138\t119.319\r\n315\t0\t0.3048\t39.6\t0.00310138\t122.779\r\n400\t0\t0.3048\t39.6\t0.00310138\t124.809\r\n500\t0\t0.3048\t39.6\t0.00310138\t126.959\r\n630\t0\t0.3048\t39.6\t0.00310138\t128.629\r\n800\t0\t0.3048\t39.6\t0.00310138\t129.099\r\n1000\t0\t0.3048\t39.6\t0.00310138\t127.899\r\n1250\t0\t0.3048\t39.6\t0.00310138\t125.499\r\n1600\t0\t0.3048\t39.6\t0.00310138\t124.049\r\n2000\t0\t0.3048\t39.6\t0.00310138\t123.689\r\n2500\t0\t0.3048\t39.6\t0.00310138\t121.399\r\n3150\t0\t0.3048\t39.6\t0.00310138\t120.319\r\n4000\t0\t0.3048\t39.6\t0.00310138\t119.229\r\n5000\t0\t0.3048\t39.6\t0.00310138\t117.789\r\n6300\t0\t0.3048\t39.6\t0.00310138\t116.229\r\n8000\t0\t0.3048\t39.6\t0.00310138\t114.779\r\n10000\t0\t0.3048\t39.6\t0.00310138\t112.139\r\n12500\t0\t0.3048\t39.6\t0.00310138\t109.619\r\n200\t0\t0.3048\t31.7\t0.00331266\t117.195\r\n250\t0\t0.3048\t31.7\t0.00331266\t118.595\r\n315\t0\t0.3048\t31.7\t0.00331266\t122.765\r\n400\t0\t0.3048\t31.7\t0.00331266\t125.045\r\n500\t0\t0.3048\t31.7\t0.00331266\t127.315\r\n630\t0\t0.3048\t31.7\t0.00331266\t129.095\r\n800\t0\t0.3048\t31.7\t0.00331266\t129.235\r\n1000\t0\t0.3048\t31.7\t0.00331266\t127.365\r\n1250\t0\t0.3048\t31.7\t0.00331266\t124.355\r\n1600\t0\t0.3048\t31.7\t0.00331266\t122.365\r\n2000\t0\t0.3048\t31.7\t0.00331266\t122.375\r\n2500\t0\t0.3048\t31.7\t0.00331266\t120.755\r\n3150\t0\t0.3048\t31.7\t0.00331266\t119.135\r\n4000\t0\t0.3048\t31.7\t0.00331266\t118.145\r\n5000\t0\t0.3048\t31.7\t0.00331266\t115.645\r\n6300\t0\t0.3048\t31.7\t0.00331266\t113.775\r\n8000\t0\t0.3048\t31.7\t0.00331266\t110.515\r\n10000\t0\t0.3048\t31.7\t0.00331266\t108.265\r\n800\t1.5\t0.3048\t71.3\t0.00336729\t127.122\r\n1000\t1.5\t0.3048\t71.3\t0.00336729\t125.992\r\n1250\t1.5\t0.3048\t71.3\t0.00336729\t125.872\r\n1600\t1.5\t0.3048\t71.3\t0.00336729\t126.632\r\n2000\t1.5\t0.3048\t71.3\t0.00336729\t126.642\r\n2500\t1.5\t0.3048\t71.3\t0.00336729\t124.512\r\n3150\t1.5\t0.3048\t71.3\t0.00336729\t123.392\r\n4000\t1.5\t0.3048\t71.3\t0.00336729\t121.762\r\n5000\t1.5\t0.3048\t71.3\t0.00336729\t119.632\r\n6300\t1.5\t0.3048\t71.3\t0.00336729\t118.122\r\n8000\t1.5\t0.3048\t71.3\t0.00336729\t115.372\r\n10000\t1.5\t0.3048\t71.3\t0.00336729\t113.492\r\n12500\t1.5\t0.3048\t71.3\t0.00336729\t109.222\r\n16000\t1.5\t0.3048\t71.3\t0.00336729\t106.582\r\n315\t1.5\t0.3048\t39.6\t0.00392107\t121.851\r\n400\t1.5\t0.3048\t39.6\t0.00392107\t124.001\r\n500\t1.5\t0.3048\t39.6\t0.00392107\t126.661\r\n630\t1.5\t0.3048\t39.6\t0.00392107\t128.311\r\n800\t1.5\t0.3048\t39.6\t0.00392107\t128.831\r\n1000\t1.5\t0.3048\t39.6\t0.00392107\t127.581\r\n1250\t1.5\t0.3048\t39.6\t0.00392107\t125.211\r\n1600\t1.5\t0.3048\t39.6\t0.00392107\t122.211\r\n2000\t1.5\t0.3048\t39.6\t0.00392107\t122.101\r\n2500\t1.5\t0.3048\t39.6\t0.00392107\t120.981\r\n3150\t1.5\t0.3048\t39.6\t0.00392107\t119.111\r\n4000\t1.5\t0.3048\t39.6\t0.00392107\t117.741\r\n5000\t1.5\t0.3048\t39.6\t0.00392107\t116.241\r\n6300\t1.5\t0.3048\t39.6\t0.00392107\t114.751\r\n8000\t1.5\t0.3048\t39.6\t0.00392107\t112.251\r\n10000\t1.5\t0.3048\t39.6\t0.00392107\t108.991\r\n12500\t1.5\t0.3048\t39.6\t0.00392107\t106.111\r\n400\t3\t0.3048\t71.3\t0.00425727\t127.564\r\n500\t3\t0.3048\t71.3\t0.00425727\t128.454\r\n630\t3\t0.3048\t71.3\t0.00425727\t129.354\r\n800\t3\t0.3048\t71.3\t0.00425727\t129.494\r\n1000\t3\t0.3048\t71.3\t0.00425727\t129.004\r\n1250\t3\t0.3048\t71.3\t0.00425727\t127.634\r\n1600\t3\t0.3048\t71.3\t0.00425727\t126.514\r\n2000\t3\t0.3048\t71.3\t0.00425727\t125.524\r\n2500\t3\t0.3048\t71.3\t0.00425727\t124.024\r\n3150\t3\t0.3048\t71.3\t0.00425727\t121.514\r\n4000\t3\t0.3048\t71.3\t0.00425727\t120.264\r\n5000\t3\t0.3048\t71.3\t0.00425727\t118.134\r\n6300\t3\t0.3048\t71.3\t0.00425727\t116.134\r\n8000\t3\t0.3048\t71.3\t0.00425727\t114.634\r\n10000\t3\t0.3048\t71.3\t0.00425727\t110.224\r\n400\t3\t0.3048\t55.5\t0.00452492\t126.159\r\n500\t3\t0.3048\t55.5\t0.00452492\t128.179\r\n630\t3\t0.3048\t55.5\t0.00452492\t129.569\r\n800\t3\t0.3048\t55.5\t0.00452492\t129.949\r\n1000\t3\t0.3048\t55.5\t0.00452492\t129.329\r\n1250\t3\t0.3048\t55.5\t0.00452492\t127.329\r\n1600\t3\t0.3048\t55.5\t0.00452492\t124.439\r\n2000\t3\t0.3048\t55.5\t0.00452492\t123.069\r\n2500\t3\t0.3048\t55.5\t0.00452492\t122.439\r\n3150\t3\t0.3048\t55.5\t0.00452492\t120.189\r\n4000\t3\t0.3048\t55.5\t0.00452492\t118.689\r\n5000\t3\t0.3048\t55.5\t0.00452492\t117.309\r\n6300\t3\t0.3048\t55.5\t0.00452492\t115.679\r\n8000\t3\t0.3048\t55.5\t0.00452492\t113.799\r\n10000\t3\t0.3048\t55.5\t0.00452492\t112.169\r\n315\t3\t0.3048\t39.6\t0.00495741\t123.312\r\n400\t3\t0.3048\t39.6\t0.00495741\t125.472\r\n500\t3\t0.3048\t39.6\t0.00495741\t127.632\r\n630\t3\t0.3048\t39.6\t0.00495741\t129.292\r\n800\t3\t0.3048\t39.6\t0.00495741\t129.552\r\n1000\t3\t0.3048\t39.6\t0.00495741\t128.312\r\n1250\t3\t0.3048\t39.6\t0.00495741\t125.802\r\n1600\t3\t0.3048\t39.6\t0.00495741\t122.782\r\n2000\t3\t0.3048\t39.6\t0.00495741\t120.532\r\n2500\t3\t0.3048\t39.6\t0.00495741\t120.162\r\n3150\t3\t0.3048\t39.6\t0.00495741\t118.922\r\n4000\t3\t0.3048\t39.6\t0.00495741\t116.792\r\n5000\t3\t0.3048\t39.6\t0.00495741\t115.792\r\n6300\t3\t0.3048\t39.6\t0.00495741\t114.042\r\n8000\t3\t0.3048\t39.6\t0.00495741\t110.652\r\n315\t3\t0.3048\t31.7\t0.00529514\t123.118\r\n400\t3\t0.3048\t31.7\t0.00529514\t125.398\r\n500\t3\t0.3048\t31.7\t0.00529514\t127.548\r\n630\t3\t0.3048\t31.7\t0.00529514\t128.698\r\n800\t3\t0.3048\t31.7\t0.00529514\t128.708\r\n1000\t3\t0.3048\t31.7\t0.00529514\t126.838\r\n1250\t3\t0.3048\t31.7\t0.00529514\t124.838\r\n1600\t3\t0.3048\t31.7\t0.00529514\t122.088\r\n2000\t3\t0.3048\t31.7\t0.00529514\t120.088\r\n2500\t3\t0.3048\t31.7\t0.00529514\t119.598\r\n3150\t3\t0.3048\t31.7\t0.00529514\t118.108\r\n4000\t3\t0.3048\t31.7\t0.00529514\t115.608\r\n5000\t3\t0.3048\t31.7\t0.00529514\t113.858\r\n6300\t3\t0.3048\t31.7\t0.00529514\t109.718\r\n250\t4\t0.3048\t71.3\t0.00497773\t126.395\r\n315\t4\t0.3048\t71.3\t0.00497773\t128.175\r\n400\t4\t0.3048\t71.3\t0.00497773\t129.575\r\n500\t4\t0.3048\t71.3\t0.00497773\t130.715\r\n630\t4\t0.3048\t71.3\t0.00497773\t131.615\r\n800\t4\t0.3048\t71.3\t0.00497773\t131.755\r\n1000\t4\t0.3048\t71.3\t0.00497773\t131.015\r\n1250\t4\t0.3048\t71.3\t0.00497773\t129.395\r\n1600\t4\t0.3048\t71.3\t0.00497773\t126.645\r\n2000\t4\t0.3048\t71.3\t0.00497773\t124.395\r\n2500\t4\t0.3048\t71.3\t0.00497773\t123.775\r\n3150\t4\t0.3048\t71.3\t0.00497773\t121.775\r\n4000\t4\t0.3048\t71.3\t0.00497773\t119.535\r\n5000\t4\t0.3048\t71.3\t0.00497773\t117.785\r\n6300\t4\t0.3048\t71.3\t0.00497773\t116.165\r\n8000\t4\t0.3048\t71.3\t0.00497773\t113.665\r\n10000\t4\t0.3048\t71.3\t0.00497773\t110.905\r\n12500\t4\t0.3048\t71.3\t0.00497773\t107.405\r\n250\t4\t0.3048\t39.6\t0.00579636\t123.543\r\n315\t4\t0.3048\t39.6\t0.00579636\t126.843\r\n400\t4\t0.3048\t39.6\t0.00579636\t128.633\r\n500\t4\t0.3048\t39.6\t0.00579636\t130.173\r\n630\t4\t0.3048\t39.6\t0.00579636\t131.073\r\n800\t4\t0.3048\t39.6\t0.00579636\t130.723\r\n1000\t4\t0.3048\t39.6\t0.00579636\t128.723\r\n1250\t4\t0.3048\t39.6\t0.00579636\t126.343\r\n1600\t4\t0.3048\t39.6\t0.00579636\t123.213\r\n2000\t4\t0.3048\t39.6\t0.00579636\t120.963\r\n2500\t4\t0.3048\t39.6\t0.00579636\t120.233\r\n3150\t4\t0.3048\t39.6\t0.00579636\t118.743\r\n4000\t4\t0.3048\t39.6\t0.00579636\t115.863\r\n5000\t4\t0.3048\t39.6\t0.00579636\t113.733\r\n1250\t0\t0.2286\t71.3\t0.00214345\t128.144\r\n1600\t0\t0.2286\t71.3\t0.00214345\t129.134\r\n2000\t0\t0.2286\t71.3\t0.00214345\t128.244\r\n2500\t0\t0.2286\t71.3\t0.00214345\t128.354\r\n3150\t0\t0.2286\t71.3\t0.00214345\t127.834\r\n4000\t0\t0.2286\t71.3\t0.00214345\t125.824\r\n5000\t0\t0.2286\t71.3\t0.00214345\t124.304\r\n6300\t0\t0.2286\t71.3\t0.00214345\t122.044\r\n8000\t0\t0.2286\t71.3\t0.00214345\t118.024\r\n10000\t0\t0.2286\t71.3\t0.00214345\t118.134\r\n12500\t0\t0.2286\t71.3\t0.00214345\t117.624\r\n16000\t0\t0.2286\t71.3\t0.00214345\t114.984\r\n20000\t0\t0.2286\t71.3\t0.00214345\t114.474\r\n315\t0\t0.2286\t55.5\t0.00229336\t119.540\r\n400\t0\t0.2286\t55.5\t0.00229336\t121.660\r\n500\t0\t0.2286\t55.5\t0.00229336\t123.780\r\n630\t0\t0.2286\t55.5\t0.00229336\t126.160\r\n800\t0\t0.2286\t55.5\t0.00229336\t127.530\r\n1000\t0\t0.2286\t55.5\t0.00229336\t128.290\r\n1250\t0\t0.2286\t55.5\t0.00229336\t127.910\r\n1600\t0\t0.2286\t55.5\t0.00229336\t126.790\r\n2000\t0\t0.2286\t55.5\t0.00229336\t126.540\r\n2500\t0\t0.2286\t55.5\t0.00229336\t126.540\r\n3150\t0\t0.2286\t55.5\t0.00229336\t125.160\r\n4000\t0\t0.2286\t55.5\t0.00229336\t123.410\r\n5000\t0\t0.2286\t55.5\t0.00229336\t122.410\r\n6300\t0\t0.2286\t55.5\t0.00229336\t118.410\r\n315\t0\t0.2286\t39.6\t0.00253511\t121.055\r\n400\t0\t0.2286\t39.6\t0.00253511\t123.565\r\n500\t0\t0.2286\t39.6\t0.00253511\t126.195\r\n630\t0\t0.2286\t39.6\t0.00253511\t128.705\r\n800\t0\t0.2286\t39.6\t0.00253511\t130.205\r\n1000\t0\t0.2286\t39.6\t0.00253511\t130.435\r\n1250\t0\t0.2286\t39.6\t0.00253511\t129.395\r\n1600\t0\t0.2286\t39.6\t0.00253511\t127.095\r\n2000\t0\t0.2286\t39.6\t0.00253511\t125.305\r\n2500\t0\t0.2286\t39.6\t0.00253511\t125.025\r\n3150\t0\t0.2286\t39.6\t0.00253511\t124.625\r\n4000\t0\t0.2286\t39.6\t0.00253511\t123.465\r\n5000\t0\t0.2286\t39.6\t0.00253511\t122.175\r\n6300\t0\t0.2286\t39.6\t0.00253511\t117.465\r\n315\t0\t0.2286\t31.7\t0.0027238\t120.595\r\n400\t0\t0.2286\t31.7\t0.0027238\t123.635\r\n500\t0\t0.2286\t31.7\t0.0027238\t126.675\r\n630\t0\t0.2286\t31.7\t0.0027238\t129.465\r\n800\t0\t0.2286\t31.7\t0.0027238\t130.725\r\n1000\t0\t0.2286\t31.7\t0.0027238\t130.595\r\n1250\t0\t0.2286\t31.7\t0.0027238\t128.805\r\n1600\t0\t0.2286\t31.7\t0.0027238\t125.625\r\n2000\t0\t0.2286\t31.7\t0.0027238\t123.455\r\n2500\t0\t0.2286\t31.7\t0.0027238\t123.445\r\n3150\t0\t0.2286\t31.7\t0.0027238\t123.445\r\n4000\t0\t0.2286\t31.7\t0.0027238\t122.035\r\n5000\t0\t0.2286\t31.7\t0.0027238\t120.505\r\n6300\t0\t0.2286\t31.7\t0.0027238\t116.815\r\n400\t2\t0.2286\t71.3\t0.00293031\t125.116\r\n500\t2\t0.2286\t71.3\t0.00293031\t126.486\r\n630\t2\t0.2286\t71.3\t0.00293031\t127.356\r\n800\t2\t0.2286\t71.3\t0.00293031\t128.216\r\n1000\t2\t0.2286\t71.3\t0.00293031\t128.956\r\n1250\t2\t0.2286\t71.3\t0.00293031\t128.816\r\n1600\t2\t0.2286\t71.3\t0.00293031\t127.796\r\n2000\t2\t0.2286\t71.3\t0.00293031\t126.896\r\n2500\t2\t0.2286\t71.3\t0.00293031\t127.006\r\n3150\t2\t0.2286\t71.3\t0.00293031\t126.116\r\n4000\t2\t0.2286\t71.3\t0.00293031\t124.086\r\n5000\t2\t0.2286\t71.3\t0.00293031\t122.816\r\n6300\t2\t0.2286\t71.3\t0.00293031\t120.786\r\n8000\t2\t0.2286\t71.3\t0.00293031\t115.996\r\n10000\t2\t0.2286\t71.3\t0.00293031\t113.086\r\n400\t2\t0.2286\t55.5\t0.00313525\t122.292\r\n500\t2\t0.2286\t55.5\t0.00313525\t124.692\r\n630\t2\t0.2286\t55.5\t0.00313525\t126.842\r\n800\t2\t0.2286\t55.5\t0.00313525\t128.492\r\n1000\t2\t0.2286\t55.5\t0.00313525\t129.002\r\n1250\t2\t0.2286\t55.5\t0.00313525\t128.762\r\n1600\t2\t0.2286\t55.5\t0.00313525\t126.752\r\n2000\t2\t0.2286\t55.5\t0.00313525\t124.612\r\n2500\t2\t0.2286\t55.5\t0.00313525\t123.862\r\n3150\t2\t0.2286\t55.5\t0.00313525\t123.742\r\n4000\t2\t0.2286\t55.5\t0.00313525\t122.232\r\n5000\t2\t0.2286\t55.5\t0.00313525\t120.472\r\n6300\t2\t0.2286\t55.5\t0.00313525\t118.712\r\n315\t2\t0.2286\t39.6\t0.00346574\t120.137\r\n400\t2\t0.2286\t39.6\t0.00346574\t122.147\r\n500\t2\t0.2286\t39.6\t0.00346574\t125.157\r\n630\t2\t0.2286\t39.6\t0.00346574\t127.417\r\n800\t2\t0.2286\t39.6\t0.00346574\t129.037\r\n1000\t2\t0.2286\t39.6\t0.00346574\t129.147\r\n1250\t2\t0.2286\t39.6\t0.00346574\t128.257\r\n1600\t2\t0.2286\t39.6\t0.00346574\t125.837\r\n2000\t2\t0.2286\t39.6\t0.00346574\t122.797\r\n2500\t2\t0.2286\t39.6\t0.00346574\t121.397\r\n3150\t2\t0.2286\t39.6\t0.00346574\t121.627\r\n4000\t2\t0.2286\t39.6\t0.00346574\t120.227\r\n5000\t2\t0.2286\t39.6\t0.00346574\t118.827\r\n6300\t2\t0.2286\t39.6\t0.00346574\t116.417\r\n315\t2\t0.2286\t31.7\t0.00372371\t120.147\r\n400\t2\t0.2286\t31.7\t0.00372371\t123.417\r\n500\t2\t0.2286\t31.7\t0.00372371\t126.677\r\n630\t2\t0.2286\t31.7\t0.00372371\t129.057\r\n800\t2\t0.2286\t31.7\t0.00372371\t130.307\r\n1000\t2\t0.2286\t31.7\t0.00372371\t130.307\r\n1250\t2\t0.2286\t31.7\t0.00372371\t128.677\r\n1600\t2\t0.2286\t31.7\t0.00372371\t125.797\r\n2000\t2\t0.2286\t31.7\t0.00372371\t123.037\r\n2500\t2\t0.2286\t31.7\t0.00372371\t121.407\r\n3150\t2\t0.2286\t31.7\t0.00372371\t121.527\r\n4000\t2\t0.2286\t31.7\t0.00372371\t120.527\r\n5000\t2\t0.2286\t31.7\t0.00372371\t118.267\r\n6300\t2\t0.2286\t31.7\t0.00372371\t115.137\r\n500\t4\t0.2286\t71.3\t0.00400603\t126.758\r\n630\t4\t0.2286\t71.3\t0.00400603\t129.038\r\n800\t4\t0.2286\t71.3\t0.00400603\t130.688\r\n1000\t4\t0.2286\t71.3\t0.00400603\t131.708\r\n1250\t4\t0.2286\t71.3\t0.00400603\t131.718\r\n1600\t4\t0.2286\t71.3\t0.00400603\t129.468\r\n2000\t4\t0.2286\t71.3\t0.00400603\t126.218\r\n2500\t4\t0.2286\t71.3\t0.00400603\t124.338\r\n3150\t4\t0.2286\t71.3\t0.00400603\t124.108\r\n4000\t4\t0.2286\t71.3\t0.00400603\t121.728\r\n5000\t4\t0.2286\t71.3\t0.00400603\t121.118\r\n6300\t4\t0.2286\t71.3\t0.00400603\t118.618\r\n8000\t4\t0.2286\t71.3\t0.00400603\t112.848\r\n10000\t4\t0.2286\t71.3\t0.00400603\t113.108\r\n12500\t4\t0.2286\t71.3\t0.00400603\t114.258\r\n16000\t4\t0.2286\t71.3\t0.00400603\t112.768\r\n20000\t4\t0.2286\t71.3\t0.00400603\t109.638\r\n400\t4\t0.2286\t55.5\t0.0042862\t123.274\r\n500\t4\t0.2286\t55.5\t0.0042862\t127.314\r\n630\t4\t0.2286\t55.5\t0.0042862\t129.964\r\n800\t4\t0.2286\t55.5\t0.0042862\t131.864\r\n1000\t4\t0.2286\t55.5\t0.0042862\t132.134\r\n1250\t4\t0.2286\t55.5\t0.0042862\t131.264\r\n1600\t4\t0.2286\t55.5\t0.0042862\t128.264\r\n2000\t4\t0.2286\t55.5\t0.0042862\t124.254\r\n2500\t4\t0.2286\t55.5\t0.0042862\t122.384\r\n3150\t4\t0.2286\t55.5\t0.0042862\t122.394\r\n4000\t4\t0.2286\t55.5\t0.0042862\t120.654\r\n5000\t4\t0.2286\t55.5\t0.0042862\t120.034\r\n6300\t4\t0.2286\t55.5\t0.0042862\t117.154\r\n8000\t4\t0.2286\t55.5\t0.0042862\t112.524\r\n315\t4\t0.2286\t39.6\t0.00473801\t122.229\r\n400\t4\t0.2286\t39.6\t0.00473801\t123.879\r\n500\t4\t0.2286\t39.6\t0.00473801\t127.039\r\n630\t4\t0.2286\t39.6\t0.00473801\t129.579\r\n800\t4\t0.2286\t39.6\t0.00473801\t130.469\r\n1000\t4\t0.2286\t39.6\t0.00473801\t129.969\r\n1250\t4\t0.2286\t39.6\t0.00473801\t128.339\r\n1600\t4\t0.2286\t39.6\t0.00473801\t125.319\r\n2000\t4\t0.2286\t39.6\t0.00473801\t121.659\r\n2500\t4\t0.2286\t39.6\t0.00473801\t119.649\r\n3150\t4\t0.2286\t39.6\t0.00473801\t120.419\r\n4000\t4\t0.2286\t39.6\t0.00473801\t119.159\r\n5000\t4\t0.2286\t39.6\t0.00473801\t117.649\r\n6300\t4\t0.2286\t39.6\t0.00473801\t114.249\r\n8000\t4\t0.2286\t39.6\t0.00473801\t113.129\r\n250\t4\t0.2286\t31.7\t0.00509068\t120.189\r\n315\t4\t0.2286\t31.7\t0.00509068\t123.609\r\n400\t4\t0.2286\t31.7\t0.00509068\t126.149\r\n500\t4\t0.2286\t31.7\t0.00509068\t128.939\r\n630\t4\t0.2286\t31.7\t0.00509068\t130.349\r\n800\t4\t0.2286\t31.7\t0.00509068\t130.869\r\n1000\t4\t0.2286\t31.7\t0.00509068\t129.869\r\n1250\t4\t0.2286\t31.7\t0.00509068\t128.119\r\n1600\t4\t0.2286\t31.7\t0.00509068\t125.229\r\n2000\t4\t0.2286\t31.7\t0.00509068\t122.089\r\n2500\t4\t0.2286\t31.7\t0.00509068\t120.209\r\n3150\t4\t0.2286\t31.7\t0.00509068\t120.229\r\n4000\t4\t0.2286\t31.7\t0.00509068\t118.859\r\n5000\t4\t0.2286\t31.7\t0.00509068\t115.969\r\n6300\t4\t0.2286\t31.7\t0.00509068\t112.699\r\n400\t5.3\t0.2286\t71.3\t0.0051942\t127.700\r\n500\t5.3\t0.2286\t71.3\t0.0051942\t129.880\r\n630\t5.3\t0.2286\t71.3\t0.0051942\t131.800\r\n800\t5.3\t0.2286\t71.3\t0.0051942\t133.480\r\n1000\t5.3\t0.2286\t71.3\t0.0051942\t134.000\r\n1250\t5.3\t0.2286\t71.3\t0.0051942\t133.380\r\n1600\t5.3\t0.2286\t71.3\t0.0051942\t130.460\r\n2000\t5.3\t0.2286\t71.3\t0.0051942\t125.890\r\n2500\t5.3\t0.2286\t71.3\t0.0051942\t123.740\r\n3150\t5.3\t0.2286\t71.3\t0.0051942\t123.120\r\n4000\t5.3\t0.2286\t71.3\t0.0051942\t120.330\r\n5000\t5.3\t0.2286\t71.3\t0.0051942\t118.050\r\n6300\t5.3\t0.2286\t71.3\t0.0051942\t116.920\r\n8000\t5.3\t0.2286\t71.3\t0.0051942\t114.900\r\n10000\t5.3\t0.2286\t71.3\t0.0051942\t111.350\r\n250\t5.3\t0.2286\t39.6\t0.00614329\t127.011\r\n315\t5.3\t0.2286\t39.6\t0.00614329\t129.691\r\n400\t5.3\t0.2286\t39.6\t0.00614329\t131.221\r\n500\t5.3\t0.2286\t39.6\t0.00614329\t132.251\r\n630\t5.3\t0.2286\t39.6\t0.00614329\t132.011\r\n800\t5.3\t0.2286\t39.6\t0.00614329\t129.491\r\n1000\t5.3\t0.2286\t39.6\t0.00614329\t125.581\r\n1250\t5.3\t0.2286\t39.6\t0.00614329\t125.721\r\n1600\t5.3\t0.2286\t39.6\t0.00614329\t123.081\r\n2000\t5.3\t0.2286\t39.6\t0.00614329\t117.911\r\n2500\t5.3\t0.2286\t39.6\t0.00614329\t116.151\r\n3150\t5.3\t0.2286\t39.6\t0.00614329\t118.441\r\n4000\t5.3\t0.2286\t39.6\t0.00614329\t115.801\r\n5000\t5.3\t0.2286\t39.6\t0.00614329\t115.311\r\n6300\t5.3\t0.2286\t39.6\t0.00614329\t112.541\r\n200\t7.3\t0.2286\t71.3\t0.0104404\t138.758\r\n250\t7.3\t0.2286\t71.3\t0.0104404\t139.918\r\n315\t7.3\t0.2286\t71.3\t0.0104404\t139.808\r\n400\t7.3\t0.2286\t71.3\t0.0104404\t139.438\r\n500\t7.3\t0.2286\t71.3\t0.0104404\t136.798\r\n630\t7.3\t0.2286\t71.3\t0.0104404\t133.768\r\n800\t7.3\t0.2286\t71.3\t0.0104404\t130.748\r\n1000\t7.3\t0.2286\t71.3\t0.0104404\t126.838\r\n1250\t7.3\t0.2286\t71.3\t0.0104404\t127.358\r\n1600\t7.3\t0.2286\t71.3\t0.0104404\t125.728\r\n2000\t7.3\t0.2286\t71.3\t0.0104404\t122.708\r\n2500\t7.3\t0.2286\t71.3\t0.0104404\t122.088\r\n3150\t7.3\t0.2286\t71.3\t0.0104404\t120.458\r\n4000\t7.3\t0.2286\t71.3\t0.0104404\t119.208\r\n5000\t7.3\t0.2286\t71.3\t0.0104404\t115.298\r\n6300\t7.3\t0.2286\t71.3\t0.0104404\t115.818\r\n200\t7.3\t0.2286\t55.5\t0.0111706\t135.234\r\n250\t7.3\t0.2286\t55.5\t0.0111706\t136.384\r\n315\t7.3\t0.2286\t55.5\t0.0111706\t136.284\r\n400\t7.3\t0.2286\t55.5\t0.0111706\t135.924\r\n500\t7.3\t0.2286\t55.5\t0.0111706\t133.174\r\n630\t7.3\t0.2286\t55.5\t0.0111706\t130.934\r\n800\t7.3\t0.2286\t55.5\t0.0111706\t128.444\r\n1000\t7.3\t0.2286\t55.5\t0.0111706\t125.194\r\n1250\t7.3\t0.2286\t55.5\t0.0111706\t125.724\r\n1600\t7.3\t0.2286\t55.5\t0.0111706\t123.354\r\n2000\t7.3\t0.2286\t55.5\t0.0111706\t120.354\r\n2500\t7.3\t0.2286\t55.5\t0.0111706\t118.994\r\n3150\t7.3\t0.2286\t55.5\t0.0111706\t117.134\r\n4000\t7.3\t0.2286\t55.5\t0.0111706\t117.284\r\n5000\t7.3\t0.2286\t55.5\t0.0111706\t113.144\r\n6300\t7.3\t0.2286\t55.5\t0.0111706\t111.534\r\n200\t7.3\t0.2286\t39.6\t0.0123481\t130.989\r\n250\t7.3\t0.2286\t39.6\t0.0123481\t131.889\r\n315\t7.3\t0.2286\t39.6\t0.0123481\t132.149\r\n400\t7.3\t0.2286\t39.6\t0.0123481\t132.039\r\n500\t7.3\t0.2286\t39.6\t0.0123481\t130.299\r\n630\t7.3\t0.2286\t39.6\t0.0123481\t128.929\r\n800\t7.3\t0.2286\t39.6\t0.0123481\t126.299\r\n1000\t7.3\t0.2286\t39.6\t0.0123481\t122.539\r\n1250\t7.3\t0.2286\t39.6\t0.0123481\t123.189\r\n1600\t7.3\t0.2286\t39.6\t0.0123481\t121.059\r\n2000\t7.3\t0.2286\t39.6\t0.0123481\t117.809\r\n2500\t7.3\t0.2286\t39.6\t0.0123481\t116.559\r\n3150\t7.3\t0.2286\t39.6\t0.0123481\t114.309\r\n4000\t7.3\t0.2286\t39.6\t0.0123481\t114.079\r\n5000\t7.3\t0.2286\t39.6\t0.0123481\t111.959\r\n6300\t7.3\t0.2286\t39.6\t0.0123481\t110.839\r\n200\t7.3\t0.2286\t31.7\t0.0132672\t128.679\r\n250\t7.3\t0.2286\t31.7\t0.0132672\t130.089\r\n315\t7.3\t0.2286\t31.7\t0.0132672\t130.239\r\n400\t7.3\t0.2286\t31.7\t0.0132672\t130.269\r\n500\t7.3\t0.2286\t31.7\t0.0132672\t128.169\r\n630\t7.3\t0.2286\t31.7\t0.0132672\t126.189\r\n800\t7.3\t0.2286\t31.7\t0.0132672\t123.209\r\n1000\t7.3\t0.2286\t31.7\t0.0132672\t119.099\r\n1250\t7.3\t0.2286\t31.7\t0.0132672\t120.509\r\n1600\t7.3\t0.2286\t31.7\t0.0132672\t119.039\r\n2000\t7.3\t0.2286\t31.7\t0.0132672\t115.309\r\n2500\t7.3\t0.2286\t31.7\t0.0132672\t114.709\r\n3150\t7.3\t0.2286\t31.7\t0.0132672\t113.229\r\n4000\t7.3\t0.2286\t31.7\t0.0132672\t112.639\r\n5000\t7.3\t0.2286\t31.7\t0.0132672\t111.029\r\n6300\t7.3\t0.2286\t31.7\t0.0132672\t110.689\r\n800\t0\t0.1524\t71.3\t0.0015988\t125.817\r\n1000\t0\t0.1524\t71.3\t0.0015988\t127.307\r\n1250\t0\t0.1524\t71.3\t0.0015988\t128.927\r\n1600\t0\t0.1524\t71.3\t0.0015988\t129.667\r\n2000\t0\t0.1524\t71.3\t0.0015988\t128.647\r\n2500\t0\t0.1524\t71.3\t0.0015988\t128.127\r\n3150\t0\t0.1524\t71.3\t0.0015988\t129.377\r\n4000\t0\t0.1524\t71.3\t0.0015988\t128.857\r\n5000\t0\t0.1524\t71.3\t0.0015988\t126.457\r\n6300\t0\t0.1524\t71.3\t0.0015988\t125.427\r\n8000\t0\t0.1524\t71.3\t0.0015988\t122.527\r\n10000\t0\t0.1524\t71.3\t0.0015988\t120.247\r\n12500\t0\t0.1524\t71.3\t0.0015988\t117.087\r\n16000\t0\t0.1524\t71.3\t0.0015988\t113.297\r\n500\t0\t0.1524\t55.5\t0.00172668\t120.573\r\n630\t0\t0.1524\t55.5\t0.00172668\t123.583\r\n800\t0\t0.1524\t55.5\t0.00172668\t126.713\r\n1000\t0\t0.1524\t55.5\t0.00172668\t128.583\r\n1250\t0\t0.1524\t55.5\t0.00172668\t129.953\r\n1600\t0\t0.1524\t55.5\t0.00172668\t130.183\r\n2000\t0\t0.1524\t55.5\t0.00172668\t129.673\r\n2500\t0\t0.1524\t55.5\t0.00172668\t127.763\r\n3150\t0\t0.1524\t55.5\t0.00172668\t127.753\r\n4000\t0\t0.1524\t55.5\t0.00172668\t127.233\r\n5000\t0\t0.1524\t55.5\t0.00172668\t125.203\r\n6300\t0\t0.1524\t55.5\t0.00172668\t123.303\r\n8000\t0\t0.1524\t55.5\t0.00172668\t121.903\r\n10000\t0\t0.1524\t55.5\t0.00172668\t119.253\r\n12500\t0\t0.1524\t55.5\t0.00172668\t117.093\r\n16000\t0\t0.1524\t55.5\t0.00172668\t112.803\r\n500\t0\t0.1524\t39.6\t0.00193287\t119.513\r\n630\t0\t0.1524\t39.6\t0.00193287\t124.403\r\n800\t0\t0.1524\t39.6\t0.00193287\t127.903\r\n1000\t0\t0.1524\t39.6\t0.00193287\t130.033\r\n1250\t0\t0.1524\t39.6\t0.00193287\t131.023\r\n1600\t0\t0.1524\t39.6\t0.00193287\t131.013\r\n2000\t0\t0.1524\t39.6\t0.00193287\t129.633\r\n2500\t0\t0.1524\t39.6\t0.00193287\t126.863\r\n3150\t0\t0.1524\t39.6\t0.00193287\t125.603\r\n4000\t0\t0.1524\t39.6\t0.00193287\t125.343\r\n5000\t0\t0.1524\t39.6\t0.00193287\t123.453\r\n6300\t0\t0.1524\t39.6\t0.00193287\t121.313\r\n8000\t0\t0.1524\t39.6\t0.00193287\t120.553\r\n10000\t0\t0.1524\t39.6\t0.00193287\t115.413\r\n500\t0\t0.1524\t31.7\t0.00209405\t121.617\r\n630\t0\t0.1524\t31.7\t0.00209405\t125.997\r\n800\t0\t0.1524\t31.7\t0.00209405\t129.117\r\n1000\t0\t0.1524\t31.7\t0.00209405\t130.987\r\n1250\t0\t0.1524\t31.7\t0.00209405\t131.467\r\n1600\t0\t0.1524\t31.7\t0.00209405\t130.817\r\n2000\t0\t0.1524\t31.7\t0.00209405\t128.907\r\n2500\t0\t0.1524\t31.7\t0.00209405\t125.867\r\n3150\t0\t0.1524\t31.7\t0.00209405\t124.207\r\n4000\t0\t0.1524\t31.7\t0.00209405\t123.807\r\n5000\t0\t0.1524\t31.7\t0.00209405\t122.397\r\n6300\t0\t0.1524\t31.7\t0.00209405\t119.737\r\n8000\t0\t0.1524\t31.7\t0.00209405\t117.957\r\n630\t2.7\t0.1524\t71.3\t0.00243851\t127.404\r\n800\t2.7\t0.1524\t71.3\t0.00243851\t127.394\r\n1000\t2.7\t0.1524\t71.3\t0.00243851\t128.774\r\n1250\t2.7\t0.1524\t71.3\t0.00243851\t130.144\r\n1600\t2.7\t0.1524\t71.3\t0.00243851\t130.644\r\n2000\t2.7\t0.1524\t71.3\t0.00243851\t130.114\r\n2500\t2.7\t0.1524\t71.3\t0.00243851\t128.334\r\n3150\t2.7\t0.1524\t71.3\t0.00243851\t127.054\r\n4000\t2.7\t0.1524\t71.3\t0.00243851\t126.534\r\n5000\t2.7\t0.1524\t71.3\t0.00243851\t124.364\r\n6300\t2.7\t0.1524\t71.3\t0.00243851\t121.944\r\n8000\t2.7\t0.1524\t71.3\t0.00243851\t120.534\r\n10000\t2.7\t0.1524\t71.3\t0.00243851\t116.724\r\n12500\t2.7\t0.1524\t71.3\t0.00243851\t113.034\r\n16000\t2.7\t0.1524\t71.3\t0.00243851\t110.364\r\n500\t2.7\t0.1524\t39.6\t0.00294804\t121.009\r\n630\t2.7\t0.1524\t39.6\t0.00294804\t125.809\r\n800\t2.7\t0.1524\t39.6\t0.00294804\t128.829\r\n1000\t2.7\t0.1524\t39.6\t0.00294804\t130.589\r\n1250\t2.7\t0.1524\t39.6\t0.00294804\t130.829\r\n1600\t2.7\t0.1524\t39.6\t0.00294804\t130.049\r\n2000\t2.7\t0.1524\t39.6\t0.00294804\t128.139\r\n2500\t2.7\t0.1524\t39.6\t0.00294804\t125.589\r\n3150\t2.7\t0.1524\t39.6\t0.00294804\t122.919\r\n4000\t2.7\t0.1524\t39.6\t0.00294804\t121.889\r\n5000\t2.7\t0.1524\t39.6\t0.00294804\t121.499\r\n6300\t2.7\t0.1524\t39.6\t0.00294804\t119.209\r\n8000\t2.7\t0.1524\t39.6\t0.00294804\t116.659\r\n10000\t2.7\t0.1524\t39.6\t0.00294804\t112.589\r\n12500\t2.7\t0.1524\t39.6\t0.00294804\t108.649\r\n400\t5.4\t0.1524\t71.3\t0.00401199\t124.121\r\n500\t5.4\t0.1524\t71.3\t0.00401199\t126.291\r\n630\t5.4\t0.1524\t71.3\t0.00401199\t128.971\r\n800\t5.4\t0.1524\t71.3\t0.00401199\t131.281\r\n1000\t5.4\t0.1524\t71.3\t0.00401199\t133.201\r\n1250\t5.4\t0.1524\t71.3\t0.00401199\t134.111\r\n1600\t5.4\t0.1524\t71.3\t0.00401199\t133.241\r\n2000\t5.4\t0.1524\t71.3\t0.00401199\t131.111\r\n2500\t5.4\t0.1524\t71.3\t0.00401199\t127.591\r\n3150\t5.4\t0.1524\t71.3\t0.00401199\t123.311\r\n4000\t5.4\t0.1524\t71.3\t0.00401199\t121.431\r\n5000\t5.4\t0.1524\t71.3\t0.00401199\t120.061\r\n6300\t5.4\t0.1524\t71.3\t0.00401199\t116.411\r\n400\t5.4\t0.1524\t55.5\t0.00433288\t126.807\r\n500\t5.4\t0.1524\t55.5\t0.00433288\t129.367\r\n630\t5.4\t0.1524\t55.5\t0.00433288\t131.807\r\n800\t5.4\t0.1524\t55.5\t0.00433288\t133.097\r\n1000\t5.4\t0.1524\t55.5\t0.00433288\t132.127\r\n1250\t5.4\t0.1524\t55.5\t0.00433288\t130.777\r\n1600\t5.4\t0.1524\t55.5\t0.00433288\t130.567\r\n2000\t5.4\t0.1524\t55.5\t0.00433288\t128.707\r\n2500\t5.4\t0.1524\t55.5\t0.00433288\t124.077\r\n3150\t5.4\t0.1524\t55.5\t0.00433288\t121.587\r\n4000\t5.4\t0.1524\t55.5\t0.00433288\t119.737\r\n5000\t5.4\t0.1524\t55.5\t0.00433288\t118.757\r\n6300\t5.4\t0.1524\t55.5\t0.00433288\t117.287\r\n8000\t5.4\t0.1524\t55.5\t0.00433288\t114.927\r\n315\t5.4\t0.1524\t39.6\t0.00485029\t125.347\r\n400\t5.4\t0.1524\t39.6\t0.00485029\t127.637\r\n500\t5.4\t0.1524\t39.6\t0.00485029\t129.937\r\n630\t5.4\t0.1524\t39.6\t0.00485029\t132.357\r\n800\t5.4\t0.1524\t39.6\t0.00485029\t132.757\r\n1000\t5.4\t0.1524\t39.6\t0.00485029\t130.507\r\n1250\t5.4\t0.1524\t39.6\t0.00485029\t127.117\r\n1600\t5.4\t0.1524\t39.6\t0.00485029\t126.267\r\n2000\t5.4\t0.1524\t39.6\t0.00485029\t124.647\r\n2500\t5.4\t0.1524\t39.6\t0.00485029\t120.497\r\n3150\t5.4\t0.1524\t39.6\t0.00485029\t119.137\r\n4000\t5.4\t0.1524\t39.6\t0.00485029\t117.137\r\n5000\t5.4\t0.1524\t39.6\t0.00485029\t117.037\r\n6300\t5.4\t0.1524\t39.6\t0.00485029\t116.677\r\n315\t5.4\t0.1524\t31.7\t0.00525474\t125.741\r\n400\t5.4\t0.1524\t31.7\t0.00525474\t127.781\r\n500\t5.4\t0.1524\t31.7\t0.00525474\t129.681\r\n630\t5.4\t0.1524\t31.7\t0.00525474\t131.471\r\n800\t5.4\t0.1524\t31.7\t0.00525474\t131.491\r\n1000\t5.4\t0.1524\t31.7\t0.00525474\t128.241\r\n1250\t5.4\t0.1524\t31.7\t0.00525474\t123.991\r\n1600\t5.4\t0.1524\t31.7\t0.00525474\t123.761\r\n2000\t5.4\t0.1524\t31.7\t0.00525474\t122.771\r\n2500\t5.4\t0.1524\t31.7\t0.00525474\t119.151\r\n3150\t5.4\t0.1524\t31.7\t0.00525474\t118.291\r\n4000\t5.4\t0.1524\t31.7\t0.00525474\t116.181\r\n5000\t5.4\t0.1524\t31.7\t0.00525474\t115.691\r\n6300\t5.4\t0.1524\t31.7\t0.00525474\t115.591\r\n315\t7.2\t0.1524\t71.3\t0.00752039\t128.713\r\n400\t7.2\t0.1524\t71.3\t0.00752039\t130.123\r\n500\t7.2\t0.1524\t71.3\t0.00752039\t132.043\r\n630\t7.2\t0.1524\t71.3\t0.00752039\t134.853\r\n800\t7.2\t0.1524\t71.3\t0.00752039\t136.023\r\n1000\t7.2\t0.1524\t71.3\t0.00752039\t134.273\r\n1250\t7.2\t0.1524\t71.3\t0.00752039\t132.513\r\n1600\t7.2\t0.1524\t71.3\t0.00752039\t130.893\r\n2000\t7.2\t0.1524\t71.3\t0.00752039\t128.643\r\n2500\t7.2\t0.1524\t71.3\t0.00752039\t124.353\r\n3150\t7.2\t0.1524\t71.3\t0.00752039\t116.783\r\n4000\t7.2\t0.1524\t71.3\t0.00752039\t119.343\r\n5000\t7.2\t0.1524\t71.3\t0.00752039\t118.343\r\n6300\t7.2\t0.1524\t71.3\t0.00752039\t116.603\r\n8000\t7.2\t0.1524\t71.3\t0.00752039\t113.333\r\n10000\t7.2\t0.1524\t71.3\t0.00752039\t110.313\r\n250\t7.2\t0.1524\t39.6\t0.00909175\t127.488\r\n315\t7.2\t0.1524\t39.6\t0.00909175\t130.558\r\n400\t7.2\t0.1524\t39.6\t0.00909175\t132.118\r\n500\t7.2\t0.1524\t39.6\t0.00909175\t132.658\r\n630\t7.2\t0.1524\t39.6\t0.00909175\t133.198\r\n800\t7.2\t0.1524\t39.6\t0.00909175\t132.358\r\n1000\t7.2\t0.1524\t39.6\t0.00909175\t128.338\r\n1250\t7.2\t0.1524\t39.6\t0.00909175\t122.428\r\n1600\t7.2\t0.1524\t39.6\t0.00909175\t120.058\r\n2000\t7.2\t0.1524\t39.6\t0.00909175\t120.228\r\n2500\t7.2\t0.1524\t39.6\t0.00909175\t117.478\r\n3150\t7.2\t0.1524\t39.6\t0.00909175\t111.818\r\n4000\t7.2\t0.1524\t39.6\t0.00909175\t114.258\r\n5000\t7.2\t0.1524\t39.6\t0.00909175\t113.288\r\n6300\t7.2\t0.1524\t39.6\t0.00909175\t112.688\r\n8000\t7.2\t0.1524\t39.6\t0.00909175\t111.588\r\n10000\t7.2\t0.1524\t39.6\t0.00909175\t110.868\r\n200\t9.9\t0.1524\t71.3\t0.0193001\t134.319\r\n250\t9.9\t0.1524\t71.3\t0.0193001\t135.329\r\n315\t9.9\t0.1524\t71.3\t0.0193001\t135.459\r\n400\t9.9\t0.1524\t71.3\t0.0193001\t135.079\r\n500\t9.9\t0.1524\t71.3\t0.0193001\t131.279\r\n630\t9.9\t0.1524\t71.3\t0.0193001\t129.889\r\n800\t9.9\t0.1524\t71.3\t0.0193001\t128.879\r\n1000\t9.9\t0.1524\t71.3\t0.0193001\t126.349\r\n1250\t9.9\t0.1524\t71.3\t0.0193001\t122.679\r\n1600\t9.9\t0.1524\t71.3\t0.0193001\t121.789\r\n2000\t9.9\t0.1524\t71.3\t0.0193001\t120.779\r\n2500\t9.9\t0.1524\t71.3\t0.0193001\t119.639\r\n3150\t9.9\t0.1524\t71.3\t0.0193001\t116.849\r\n4000\t9.9\t0.1524\t71.3\t0.0193001\t115.079\r\n5000\t9.9\t0.1524\t71.3\t0.0193001\t114.569\r\n6300\t9.9\t0.1524\t71.3\t0.0193001\t112.039\r\n200\t9.9\t0.1524\t55.5\t0.0208438\t131.955\r\n250\t9.9\t0.1524\t55.5\t0.0208438\t133.235\r\n315\t9.9\t0.1524\t55.5\t0.0208438\t132.355\r\n400\t9.9\t0.1524\t55.5\t0.0208438\t131.605\r\n500\t9.9\t0.1524\t55.5\t0.0208438\t127.815\r\n630\t9.9\t0.1524\t55.5\t0.0208438\t127.315\r\n800\t9.9\t0.1524\t55.5\t0.0208438\t126.565\r\n1000\t9.9\t0.1524\t55.5\t0.0208438\t124.665\r\n1250\t9.9\t0.1524\t55.5\t0.0208438\t121.635\r\n1600\t9.9\t0.1524\t55.5\t0.0208438\t119.875\r\n2000\t9.9\t0.1524\t55.5\t0.0208438\t119.505\r\n2500\t9.9\t0.1524\t55.5\t0.0208438\t118.365\r\n3150\t9.9\t0.1524\t55.5\t0.0208438\t115.085\r\n4000\t9.9\t0.1524\t55.5\t0.0208438\t112.945\r\n5000\t9.9\t0.1524\t55.5\t0.0208438\t112.065\r\n6300\t9.9\t0.1524\t55.5\t0.0208438\t110.555\r\n200\t9.9\t0.1524\t39.6\t0.0233328\t127.315\r\n250\t9.9\t0.1524\t39.6\t0.0233328\t128.335\r\n315\t9.9\t0.1524\t39.6\t0.0233328\t128.595\r\n400\t9.9\t0.1524\t39.6\t0.0233328\t128.345\r\n500\t9.9\t0.1524\t39.6\t0.0233328\t126.835\r\n630\t9.9\t0.1524\t39.6\t0.0233328\t126.465\r\n800\t9.9\t0.1524\t39.6\t0.0233328\t126.345\r\n1000\t9.9\t0.1524\t39.6\t0.0233328\t123.835\r\n1250\t9.9\t0.1524\t39.6\t0.0233328\t120.555\r\n1600\t9.9\t0.1524\t39.6\t0.0233328\t118.545\r\n2000\t9.9\t0.1524\t39.6\t0.0233328\t117.925\r\n2500\t9.9\t0.1524\t39.6\t0.0233328\t116.295\r\n3150\t9.9\t0.1524\t39.6\t0.0233328\t113.525\r\n4000\t9.9\t0.1524\t39.6\t0.0233328\t112.265\r\n5000\t9.9\t0.1524\t39.6\t0.0233328\t111.135\r\n6300\t9.9\t0.1524\t39.6\t0.0233328\t109.885\r\n200\t9.9\t0.1524\t31.7\t0.0252785\t127.299\r\n250\t9.9\t0.1524\t31.7\t0.0252785\t128.559\r\n315\t9.9\t0.1524\t31.7\t0.0252785\t128.809\r\n400\t9.9\t0.1524\t31.7\t0.0252785\t128.939\r\n500\t9.9\t0.1524\t31.7\t0.0252785\t127.179\r\n630\t9.9\t0.1524\t31.7\t0.0252785\t126.049\r\n800\t9.9\t0.1524\t31.7\t0.0252785\t125.539\r\n1000\t9.9\t0.1524\t31.7\t0.0252785\t122.149\r\n1250\t9.9\t0.1524\t31.7\t0.0252785\t118.619\r\n1600\t9.9\t0.1524\t31.7\t0.0252785\t117.119\r\n2000\t9.9\t0.1524\t31.7\t0.0252785\t116.859\r\n2500\t9.9\t0.1524\t31.7\t0.0252785\t114.729\r\n3150\t9.9\t0.1524\t31.7\t0.0252785\t112.209\r\n4000\t9.9\t0.1524\t31.7\t0.0252785\t111.459\r\n5000\t9.9\t0.1524\t31.7\t0.0252785\t109.949\r\n6300\t9.9\t0.1524\t31.7\t0.0252785\t108.689\r\n200\t12.6\t0.1524\t71.3\t0.0483159\t128.354\r\n250\t12.6\t0.1524\t71.3\t0.0483159\t129.744\r\n315\t12.6\t0.1524\t71.3\t0.0483159\t128.484\r\n400\t12.6\t0.1524\t71.3\t0.0483159\t127.094\r\n500\t12.6\t0.1524\t71.3\t0.0483159\t121.664\r\n630\t12.6\t0.1524\t71.3\t0.0483159\t123.304\r\n800\t12.6\t0.1524\t71.3\t0.0483159\t123.054\r\n1000\t12.6\t0.1524\t71.3\t0.0483159\t122.044\r\n1250\t12.6\t0.1524\t71.3\t0.0483159\t120.154\r\n1600\t12.6\t0.1524\t71.3\t0.0483159\t120.534\r\n2000\t12.6\t0.1524\t71.3\t0.0483159\t117.504\r\n2500\t12.6\t0.1524\t71.3\t0.0483159\t115.234\r\n3150\t12.6\t0.1524\t71.3\t0.0483159\t113.334\r\n4000\t12.6\t0.1524\t71.3\t0.0483159\t108.034\r\n5000\t12.6\t0.1524\t71.3\t0.0483159\t108.034\r\n6300\t12.6\t0.1524\t71.3\t0.0483159\t107.284\r\n200\t12.6\t0.1524\t39.6\t0.0584113\t114.750\r\n250\t12.6\t0.1524\t39.6\t0.0584113\t115.890\r\n315\t12.6\t0.1524\t39.6\t0.0584113\t116.020\r\n400\t12.6\t0.1524\t39.6\t0.0584113\t115.910\r\n500\t12.6\t0.1524\t39.6\t0.0584113\t114.900\r\n630\t12.6\t0.1524\t39.6\t0.0584113\t116.550\r\n800\t12.6\t0.1524\t39.6\t0.0584113\t116.560\r\n1000\t12.6\t0.1524\t39.6\t0.0584113\t114.670\r\n1250\t12.6\t0.1524\t39.6\t0.0584113\t112.160\r\n1600\t12.6\t0.1524\t39.6\t0.0584113\t110.780\r\n2000\t12.6\t0.1524\t39.6\t0.0584113\t109.520\r\n2500\t12.6\t0.1524\t39.6\t0.0584113\t106.880\r\n3150\t12.6\t0.1524\t39.6\t0.0584113\t106.260\r\n4000\t12.6\t0.1524\t39.6\t0.0584113\t104.500\r\n5000\t12.6\t0.1524\t39.6\t0.0584113\t104.130\r\n6300\t12.6\t0.1524\t39.6\t0.0584113\t103.380\r\n800\t0\t0.0508\t71.3\t0.000740478\t130.960\r\n1000\t0\t0.0508\t71.3\t0.000740478\t129.450\r\n1250\t0\t0.0508\t71.3\t0.000740478\t128.560\r\n1600\t0\t0.0508\t71.3\t0.000740478\t129.680\r\n2000\t0\t0.0508\t71.3\t0.000740478\t131.060\r\n2500\t0\t0.0508\t71.3\t0.000740478\t131.310\r\n3150\t0\t0.0508\t71.3\t0.000740478\t135.070\r\n4000\t0\t0.0508\t71.3\t0.000740478\t134.430\r\n5000\t0\t0.0508\t71.3\t0.000740478\t134.430\r\n6300\t0\t0.0508\t71.3\t0.000740478\t133.040\r\n8000\t0\t0.0508\t71.3\t0.000740478\t130.890\r\n10000\t0\t0.0508\t71.3\t0.000740478\t128.740\r\n12500\t0\t0.0508\t71.3\t0.000740478\t125.220\r\n800\t0\t0.0508\t55.5\t0.00076193\t124.336\r\n1000\t0\t0.0508\t55.5\t0.00076193\t125.586\r\n1250\t0\t0.0508\t55.5\t0.00076193\t127.076\r\n1600\t0\t0.0508\t55.5\t0.00076193\t128.576\r\n2000\t0\t0.0508\t55.5\t0.00076193\t131.456\r\n2500\t0\t0.0508\t55.5\t0.00076193\t133.956\r\n3150\t0\t0.0508\t55.5\t0.00076193\t134.826\r\n4000\t0\t0.0508\t55.5\t0.00076193\t134.946\r\n5000\t0\t0.0508\t55.5\t0.00076193\t134.556\r\n6300\t0\t0.0508\t55.5\t0.00076193\t132.796\r\n8000\t0\t0.0508\t55.5\t0.00076193\t130.156\r\n10000\t0\t0.0508\t55.5\t0.00076193\t127.636\r\n12500\t0\t0.0508\t55.5\t0.00076193\t125.376\r\n800\t0\t0.0508\t39.6\t0.000791822\t126.508\r\n1000\t0\t0.0508\t39.6\t0.000791822\t127.638\r\n1250\t0\t0.0508\t39.6\t0.000791822\t129.148\r\n1600\t0\t0.0508\t39.6\t0.000791822\t130.908\r\n2000\t0\t0.0508\t39.6\t0.000791822\t132.918\r\n2500\t0\t0.0508\t39.6\t0.000791822\t134.938\r\n3150\t0\t0.0508\t39.6\t0.000791822\t135.938\r\n4000\t0\t0.0508\t39.6\t0.000791822\t135.308\r\n5000\t0\t0.0508\t39.6\t0.000791822\t134.308\r\n6300\t0\t0.0508\t39.6\t0.000791822\t131.918\r\n8000\t0\t0.0508\t39.6\t0.000791822\t128.518\r\n10000\t0\t0.0508\t39.6\t0.000791822\t125.998\r\n12500\t0\t0.0508\t39.6\t0.000791822\t123.988\r\n800\t0\t0.0508\t31.7\t0.000812164\t122.790\r\n1000\t0\t0.0508\t31.7\t0.000812164\t126.780\r\n1250\t0\t0.0508\t31.7\t0.000812164\t129.270\r\n1600\t0\t0.0508\t31.7\t0.000812164\t131.010\r\n2000\t0\t0.0508\t31.7\t0.000812164\t133.010\r\n2500\t0\t0.0508\t31.7\t0.000812164\t134.870\r\n3150\t0\t0.0508\t31.7\t0.000812164\t135.490\r\n4000\t0\t0.0508\t31.7\t0.000812164\t134.110\r\n5000\t0\t0.0508\t31.7\t0.000812164\t133.230\r\n6300\t0\t0.0508\t31.7\t0.000812164\t130.340\r\n8000\t0\t0.0508\t31.7\t0.000812164\t126.590\r\n10000\t0\t0.0508\t31.7\t0.000812164\t122.450\r\n12500\t0\t0.0508\t31.7\t0.000812164\t119.070\r\n1600\t4.2\t0.0508\t71.3\t0.00142788\t124.318\r\n2000\t4.2\t0.0508\t71.3\t0.00142788\t129.848\r\n2500\t4.2\t0.0508\t71.3\t0.00142788\t131.978\r\n3150\t4.2\t0.0508\t71.3\t0.00142788\t133.728\r\n4000\t4.2\t0.0508\t71.3\t0.00142788\t133.598\r\n5000\t4.2\t0.0508\t71.3\t0.00142788\t132.828\r\n6300\t4.2\t0.0508\t71.3\t0.00142788\t129.308\r\n8000\t4.2\t0.0508\t71.3\t0.00142788\t125.268\r\n10000\t4.2\t0.0508\t71.3\t0.00142788\t121.238\r\n12500\t4.2\t0.0508\t71.3\t0.00142788\t117.328\r\n1000\t4.2\t0.0508\t39.6\t0.00152689\t125.647\r\n1250\t4.2\t0.0508\t39.6\t0.00152689\t128.427\r\n1600\t4.2\t0.0508\t39.6\t0.00152689\t130.197\r\n2000\t4.2\t0.0508\t39.6\t0.00152689\t132.587\r\n2500\t4.2\t0.0508\t39.6\t0.00152689\t133.847\r\n3150\t4.2\t0.0508\t39.6\t0.00152689\t133.587\r\n4000\t4.2\t0.0508\t39.6\t0.00152689\t131.807\r\n5000\t4.2\t0.0508\t39.6\t0.00152689\t129.777\r\n6300\t4.2\t0.0508\t39.6\t0.00152689\t125.717\r\n8000\t4.2\t0.0508\t39.6\t0.00152689\t120.397\r\n10000\t4.2\t0.0508\t39.6\t0.00152689\t116.967\r\n800\t8.4\t0.0508\t71.3\t0.00529514\t127.556\r\n1000\t8.4\t0.0508\t71.3\t0.00529514\t129.946\r\n1250\t8.4\t0.0508\t71.3\t0.00529514\t132.086\r\n1600\t8.4\t0.0508\t71.3\t0.00529514\t133.846\r\n2000\t8.4\t0.0508\t71.3\t0.00529514\t134.476\r\n2500\t8.4\t0.0508\t71.3\t0.00529514\t134.226\r\n3150\t8.4\t0.0508\t71.3\t0.00529514\t131.966\r\n4000\t8.4\t0.0508\t71.3\t0.00529514\t126.926\r\n5000\t8.4\t0.0508\t71.3\t0.00529514\t121.146\r\n400\t8.4\t0.0508\t55.5\t0.00544854\t121.582\r\n500\t8.4\t0.0508\t55.5\t0.00544854\t123.742\r\n630\t8.4\t0.0508\t55.5\t0.00544854\t126.152\r\n800\t8.4\t0.0508\t55.5\t0.00544854\t128.562\r\n1000\t8.4\t0.0508\t55.5\t0.00544854\t130.722\r\n1250\t8.4\t0.0508\t55.5\t0.00544854\t132.252\r\n1600\t8.4\t0.0508\t55.5\t0.00544854\t133.032\r\n2000\t8.4\t0.0508\t55.5\t0.00544854\t133.042\r\n2500\t8.4\t0.0508\t55.5\t0.00544854\t131.542\r\n3150\t8.4\t0.0508\t55.5\t0.00544854\t128.402\r\n4000\t8.4\t0.0508\t55.5\t0.00544854\t122.612\r\n5000\t8.4\t0.0508\t55.5\t0.00544854\t115.812\r\n400\t8.4\t0.0508\t39.6\t0.00566229\t120.015\r\n500\t8.4\t0.0508\t39.6\t0.00566229\t122.905\r\n630\t8.4\t0.0508\t39.6\t0.00566229\t126.045\r\n800\t8.4\t0.0508\t39.6\t0.00566229\t128.435\r\n1000\t8.4\t0.0508\t39.6\t0.00566229\t130.195\r\n1250\t8.4\t0.0508\t39.6\t0.00566229\t131.205\r\n1600\t8.4\t0.0508\t39.6\t0.00566229\t130.965\r\n2000\t8.4\t0.0508\t39.6\t0.00566229\t129.965\r\n2500\t8.4\t0.0508\t39.6\t0.00566229\t127.465\r\n3150\t8.4\t0.0508\t39.6\t0.00566229\t123.965\r\n4000\t8.4\t0.0508\t39.6\t0.00566229\t118.955\r\n400\t8.4\t0.0508\t31.7\t0.00580776\t120.076\r\n500\t8.4\t0.0508\t31.7\t0.00580776\t122.966\r\n630\t8.4\t0.0508\t31.7\t0.00580776\t125.856\r\n800\t8.4\t0.0508\t31.7\t0.00580776\t128.246\r\n1000\t8.4\t0.0508\t31.7\t0.00580776\t129.516\r\n1250\t8.4\t0.0508\t31.7\t0.00580776\t130.156\r\n1600\t8.4\t0.0508\t31.7\t0.00580776\t129.296\r\n2000\t8.4\t0.0508\t31.7\t0.00580776\t127.686\r\n2500\t8.4\t0.0508\t31.7\t0.00580776\t125.576\r\n3150\t8.4\t0.0508\t31.7\t0.00580776\t122.086\r\n4000\t8.4\t0.0508\t31.7\t0.00580776\t118.106\r\n200\t11.2\t0.0508\t71.3\t0.014072\t125.941\r\n250\t11.2\t0.0508\t71.3\t0.014072\t127.101\r\n315\t11.2\t0.0508\t71.3\t0.014072\t128.381\r\n400\t11.2\t0.0508\t71.3\t0.014072\t129.281\r\n500\t11.2\t0.0508\t71.3\t0.014072\t130.311\r\n630\t11.2\t0.0508\t71.3\t0.014072\t133.611\r\n800\t11.2\t0.0508\t71.3\t0.014072\t136.031\r\n1000\t11.2\t0.0508\t71.3\t0.014072\t136.941\r\n1250\t11.2\t0.0508\t71.3\t0.014072\t136.191\r\n1600\t11.2\t0.0508\t71.3\t0.014072\t135.191\r\n2000\t11.2\t0.0508\t71.3\t0.014072\t133.311\r\n2500\t11.2\t0.0508\t71.3\t0.014072\t130.541\r\n3150\t11.2\t0.0508\t71.3\t0.014072\t127.141\r\n4000\t11.2\t0.0508\t71.3\t0.014072\t122.471\r\n200\t11.2\t0.0508\t39.6\t0.0150478\t125.010\r\n250\t11.2\t0.0508\t39.6\t0.0150478\t126.430\r\n315\t11.2\t0.0508\t39.6\t0.0150478\t128.990\r\n400\t11.2\t0.0508\t39.6\t0.0150478\t130.670\r\n500\t11.2\t0.0508\t39.6\t0.0150478\t131.960\r\n630\t11.2\t0.0508\t39.6\t0.0150478\t133.130\r\n800\t11.2\t0.0508\t39.6\t0.0150478\t133.790\r\n1000\t11.2\t0.0508\t39.6\t0.0150478\t132.430\r\n1250\t11.2\t0.0508\t39.6\t0.0150478\t130.050\r\n1600\t11.2\t0.0508\t39.6\t0.0150478\t126.540\r\n2000\t11.2\t0.0508\t39.6\t0.0150478\t124.420\r\n2500\t11.2\t0.0508\t39.6\t0.0150478\t122.170\r\n3150\t11.2\t0.0508\t39.6\t0.0150478\t119.670\r\n4000\t11.2\t0.0508\t39.6\t0.0150478\t115.520\r\n200\t15.4\t0.0508\t71.3\t0.0264269\t123.595\r\n250\t15.4\t0.0508\t71.3\t0.0264269\t124.835\r\n315\t15.4\t0.0508\t71.3\t0.0264269\t126.195\r\n400\t15.4\t0.0508\t71.3\t0.0264269\t126.805\r\n500\t15.4\t0.0508\t71.3\t0.0264269\t127.285\r\n630\t15.4\t0.0508\t71.3\t0.0264269\t129.645\r\n800\t15.4\t0.0508\t71.3\t0.0264269\t131.515\r\n1000\t15.4\t0.0508\t71.3\t0.0264269\t131.865\r\n1250\t15.4\t0.0508\t71.3\t0.0264269\t130.845\r\n1600\t15.4\t0.0508\t71.3\t0.0264269\t130.065\r\n2000\t15.4\t0.0508\t71.3\t0.0264269\t129.285\r\n2500\t15.4\t0.0508\t71.3\t0.0264269\t127.625\r\n3150\t15.4\t0.0508\t71.3\t0.0264269\t125.715\r\n4000\t15.4\t0.0508\t71.3\t0.0264269\t122.675\r\n5000\t15.4\t0.0508\t71.3\t0.0264269\t119.135\r\n6300\t15.4\t0.0508\t71.3\t0.0264269\t115.215\r\n8000\t15.4\t0.0508\t71.3\t0.0264269\t112.675\r\n200\t15.4\t0.0508\t55.5\t0.0271925\t122.940\r\n250\t15.4\t0.0508\t55.5\t0.0271925\t124.170\r\n315\t15.4\t0.0508\t55.5\t0.0271925\t125.390\r\n400\t15.4\t0.0508\t55.5\t0.0271925\t126.500\r\n500\t15.4\t0.0508\t55.5\t0.0271925\t127.220\r\n630\t15.4\t0.0508\t55.5\t0.0271925\t129.330\r\n800\t15.4\t0.0508\t55.5\t0.0271925\t130.430\r\n1000\t15.4\t0.0508\t55.5\t0.0271925\t130.400\r\n1250\t15.4\t0.0508\t55.5\t0.0271925\t130.000\r\n1600\t15.4\t0.0508\t55.5\t0.0271925\t128.200\r\n2000\t15.4\t0.0508\t55.5\t0.0271925\t127.040\r\n2500\t15.4\t0.0508\t55.5\t0.0271925\t125.630\r\n3150\t15.4\t0.0508\t55.5\t0.0271925\t123.460\r\n4000\t15.4\t0.0508\t55.5\t0.0271925\t120.920\r\n5000\t15.4\t0.0508\t55.5\t0.0271925\t117.110\r\n6300\t15.4\t0.0508\t55.5\t0.0271925\t112.930\r\n200\t15.4\t0.0508\t39.6\t0.0282593\t121.783\r\n250\t15.4\t0.0508\t39.6\t0.0282593\t122.893\r\n315\t15.4\t0.0508\t39.6\t0.0282593\t124.493\r\n400\t15.4\t0.0508\t39.6\t0.0282593\t125.353\r\n500\t15.4\t0.0508\t39.6\t0.0282593\t125.963\r\n630\t15.4\t0.0508\t39.6\t0.0282593\t127.443\r\n800\t15.4\t0.0508\t39.6\t0.0282593\t128.423\r\n1000\t15.4\t0.0508\t39.6\t0.0282593\t127.893\r\n1250\t15.4\t0.0508\t39.6\t0.0282593\t126.743\r\n1600\t15.4\t0.0508\t39.6\t0.0282593\t124.843\r\n2000\t15.4\t0.0508\t39.6\t0.0282593\t123.443\r\n2500\t15.4\t0.0508\t39.6\t0.0282593\t122.413\r\n3150\t15.4\t0.0508\t39.6\t0.0282593\t120.513\r\n4000\t15.4\t0.0508\t39.6\t0.0282593\t118.113\r\n5000\t15.4\t0.0508\t39.6\t0.0282593\t114.453\r\n6300\t15.4\t0.0508\t39.6\t0.0282593\t109.663\r\n200\t15.4\t0.0508\t31.7\t0.0289853\t119.975\r\n250\t15.4\t0.0508\t31.7\t0.0289853\t121.225\r\n315\t15.4\t0.0508\t31.7\t0.0289853\t122.845\r\n400\t15.4\t0.0508\t31.7\t0.0289853\t123.705\r\n500\t15.4\t0.0508\t31.7\t0.0289853\t123.695\r\n630\t15.4\t0.0508\t31.7\t0.0289853\t124.685\r\n800\t15.4\t0.0508\t31.7\t0.0289853\t125.555\r\n1000\t15.4\t0.0508\t31.7\t0.0289853\t124.525\r\n1250\t15.4\t0.0508\t31.7\t0.0289853\t123.255\r\n1600\t15.4\t0.0508\t31.7\t0.0289853\t121.485\r\n2000\t15.4\t0.0508\t31.7\t0.0289853\t120.835\r\n2500\t15.4\t0.0508\t31.7\t0.0289853\t119.945\r\n3150\t15.4\t0.0508\t31.7\t0.0289853\t118.045\r\n4000\t15.4\t0.0508\t31.7\t0.0289853\t115.635\r\n5000\t15.4\t0.0508\t31.7\t0.0289853\t112.355\r\n6300\t15.4\t0.0508\t31.7\t0.0289853\t108.185\r\n200\t19.7\t0.0508\t71.3\t0.0341183\t118.005\r\n250\t19.7\t0.0508\t71.3\t0.0341183\t119.115\r\n315\t19.7\t0.0508\t71.3\t0.0341183\t121.235\r\n400\t19.7\t0.0508\t71.3\t0.0341183\t123.865\r\n500\t19.7\t0.0508\t71.3\t0.0341183\t126.995\r\n630\t19.7\t0.0508\t71.3\t0.0341183\t128.365\r\n800\t19.7\t0.0508\t71.3\t0.0341183\t124.555\r\n1000\t19.7\t0.0508\t71.3\t0.0341183\t121.885\r\n1250\t19.7\t0.0508\t71.3\t0.0341183\t121.485\r\n1600\t19.7\t0.0508\t71.3\t0.0341183\t120.575\r\n2000\t19.7\t0.0508\t71.3\t0.0341183\t120.055\r\n2500\t19.7\t0.0508\t71.3\t0.0341183\t118.385\r\n3150\t19.7\t0.0508\t71.3\t0.0341183\t116.225\r\n4000\t19.7\t0.0508\t71.3\t0.0341183\t113.045\r\n200\t19.7\t0.0508\t39.6\t0.036484\t125.974\r\n250\t19.7\t0.0508\t39.6\t0.036484\t127.224\r\n315\t19.7\t0.0508\t39.6\t0.036484\t129.864\r\n400\t19.7\t0.0508\t39.6\t0.036484\t130.614\r\n500\t19.7\t0.0508\t39.6\t0.036484\t128.444\r\n630\t19.7\t0.0508\t39.6\t0.036484\t120.324\r\n800\t19.7\t0.0508\t39.6\t0.036484\t119.174\r\n1000\t19.7\t0.0508\t39.6\t0.036484\t118.904\r\n1250\t19.7\t0.0508\t39.6\t0.036484\t118.634\r\n1600\t19.7\t0.0508\t39.6\t0.036484\t117.604\r\n2000\t19.7\t0.0508\t39.6\t0.036484\t117.724\r\n2500\t19.7\t0.0508\t39.6\t0.036484\t116.184\r\n3150\t19.7\t0.0508\t39.6\t0.036484\t113.004\r\n4000\t19.7\t0.0508\t39.6\t0.036484\t108.684\r\n2500\t0\t0.0254\t71.3\t0.000400682\t133.707\r\n3150\t0\t0.0254\t71.3\t0.000400682\t137.007\r\n4000\t0\t0.0254\t71.3\t0.000400682\t138.557\r\n5000\t0\t0.0254\t71.3\t0.000400682\t136.837\r\n6300\t0\t0.0254\t71.3\t0.000400682\t134.987\r\n8000\t0\t0.0254\t71.3\t0.000400682\t129.867\r\n10000\t0\t0.0254\t71.3\t0.000400682\t130.787\r\n12500\t0\t0.0254\t71.3\t0.000400682\t133.207\r\n16000\t0\t0.0254\t71.3\t0.000400682\t130.477\r\n20000\t0\t0.0254\t71.3\t0.000400682\t123.217\r\n2000\t0\t0.0254\t55.5\t0.00041229\t127.623\r\n2500\t0\t0.0254\t55.5\t0.00041229\t130.073\r\n3150\t0\t0.0254\t55.5\t0.00041229\t130.503\r\n4000\t0\t0.0254\t55.5\t0.00041229\t133.223\r\n5000\t0\t0.0254\t55.5\t0.00041229\t135.803\r\n6300\t0\t0.0254\t55.5\t0.00041229\t136.103\r\n8000\t0\t0.0254\t55.5\t0.00041229\t136.163\r\n10000\t0\t0.0254\t55.5\t0.00041229\t134.563\r\n12500\t0\t0.0254\t55.5\t0.00041229\t131.453\r\n16000\t0\t0.0254\t55.5\t0.00041229\t125.683\r\n20000\t0\t0.0254\t55.5\t0.00041229\t121.933\r\n1600\t0\t0.0254\t39.6\t0.000428464\t124.156\r\n2000\t0\t0.0254\t39.6\t0.000428464\t130.026\r\n2500\t0\t0.0254\t39.6\t0.000428464\t131.836\r\n3150\t0\t0.0254\t39.6\t0.000428464\t133.276\r\n4000\t0\t0.0254\t39.6\t0.000428464\t135.346\r\n5000\t0\t0.0254\t39.6\t0.000428464\t136.536\r\n6300\t0\t0.0254\t39.6\t0.000428464\t136.826\r\n8000\t0\t0.0254\t39.6\t0.000428464\t135.866\r\n10000\t0\t0.0254\t39.6\t0.000428464\t133.376\r\n12500\t0\t0.0254\t39.6\t0.000428464\t129.116\r\n16000\t0\t0.0254\t39.6\t0.000428464\t124.986\r\n1000\t0\t0.0254\t31.7\t0.000439472\t125.127\r\n1250\t0\t0.0254\t31.7\t0.000439472\t127.947\r\n1600\t0\t0.0254\t31.7\t0.000439472\t129.267\r\n2000\t0\t0.0254\t31.7\t0.000439472\t130.697\r\n2500\t0\t0.0254\t31.7\t0.000439472\t132.897\r\n3150\t0\t0.0254\t31.7\t0.000439472\t135.227\r\n4000\t0\t0.0254\t31.7\t0.000439472\t137.047\r\n5000\t0\t0.0254\t31.7\t0.000439472\t138.607\r\n6300\t0\t0.0254\t31.7\t0.000439472\t138.537\r\n8000\t0\t0.0254\t31.7\t0.000439472\t137.207\r\n10000\t0\t0.0254\t31.7\t0.000439472\t134.227\r\n12500\t0\t0.0254\t31.7\t0.000439472\t128.977\r\n16000\t0\t0.0254\t31.7\t0.000439472\t125.627\r\n2000\t4.8\t0.0254\t71.3\t0.000848633\t128.398\r\n2500\t4.8\t0.0254\t71.3\t0.000848633\t130.828\r\n3150\t4.8\t0.0254\t71.3\t0.000848633\t133.378\r\n4000\t4.8\t0.0254\t71.3\t0.000848633\t134.928\r\n5000\t4.8\t0.0254\t71.3\t0.000848633\t135.468\r\n6300\t4.8\t0.0254\t71.3\t0.000848633\t134.498\r\n8000\t4.8\t0.0254\t71.3\t0.000848633\t131.518\r\n10000\t4.8\t0.0254\t71.3\t0.000848633\t127.398\r\n12500\t4.8\t0.0254\t71.3\t0.000848633\t127.688\r\n16000\t4.8\t0.0254\t71.3\t0.000848633\t124.208\r\n20000\t4.8\t0.0254\t71.3\t0.000848633\t119.708\r\n1600\t4.8\t0.0254\t55.5\t0.000873218\t121.474\r\n2000\t4.8\t0.0254\t55.5\t0.000873218\t125.054\r\n2500\t4.8\t0.0254\t55.5\t0.000873218\t129.144\r\n3150\t4.8\t0.0254\t55.5\t0.000873218\t132.354\r\n4000\t4.8\t0.0254\t55.5\t0.000873218\t133.924\r\n5000\t4.8\t0.0254\t55.5\t0.000873218\t135.484\r\n6300\t4.8\t0.0254\t55.5\t0.000873218\t135.164\r\n8000\t4.8\t0.0254\t55.5\t0.000873218\t132.184\r\n10000\t4.8\t0.0254\t55.5\t0.000873218\t126.944\r\n12500\t4.8\t0.0254\t55.5\t0.000873218\t125.094\r\n16000\t4.8\t0.0254\t55.5\t0.000873218\t124.394\r\n20000\t4.8\t0.0254\t55.5\t0.000873218\t121.284\r\n500\t4.8\t0.0254\t39.6\t0.000907475\t116.366\r\n630\t4.8\t0.0254\t39.6\t0.000907475\t118.696\r\n800\t4.8\t0.0254\t39.6\t0.000907475\t120.766\r\n1000\t4.8\t0.0254\t39.6\t0.000907475\t122.956\r\n1250\t4.8\t0.0254\t39.6\t0.000907475\t125.026\r\n1600\t4.8\t0.0254\t39.6\t0.000907475\t125.966\r\n2000\t4.8\t0.0254\t39.6\t0.000907475\t128.916\r\n2500\t4.8\t0.0254\t39.6\t0.000907475\t131.236\r\n3150\t4.8\t0.0254\t39.6\t0.000907475\t133.436\r\n4000\t4.8\t0.0254\t39.6\t0.000907475\t134.996\r\n5000\t4.8\t0.0254\t39.6\t0.000907475\t135.426\r\n6300\t4.8\t0.0254\t39.6\t0.000907475\t134.336\r\n8000\t4.8\t0.0254\t39.6\t0.000907475\t131.346\r\n10000\t4.8\t0.0254\t39.6\t0.000907475\t126.066\r\n500\t4.8\t0.0254\t31.7\t0.000930789\t116.128\r\n630\t4.8\t0.0254\t31.7\t0.000930789\t120.078\r\n800\t4.8\t0.0254\t31.7\t0.000930789\t122.648\r\n1000\t4.8\t0.0254\t31.7\t0.000930789\t125.348\r\n1250\t4.8\t0.0254\t31.7\t0.000930789\t127.408\r\n1600\t4.8\t0.0254\t31.7\t0.000930789\t128.718\r\n2000\t4.8\t0.0254\t31.7\t0.000930789\t130.148\r\n2500\t4.8\t0.0254\t31.7\t0.000930789\t132.588\r\n3150\t4.8\t0.0254\t31.7\t0.000930789\t134.268\r\n4000\t4.8\t0.0254\t31.7\t0.000930789\t135.328\r\n5000\t4.8\t0.0254\t31.7\t0.000930789\t135.248\r\n6300\t4.8\t0.0254\t31.7\t0.000930789\t132.898\r\n8000\t4.8\t0.0254\t31.7\t0.000930789\t127.008\r\n630\t9.5\t0.0254\t71.3\t0.00420654\t125.726\r\n800\t9.5\t0.0254\t71.3\t0.00420654\t127.206\r\n1000\t9.5\t0.0254\t71.3\t0.00420654\t129.556\r\n1250\t9.5\t0.0254\t71.3\t0.00420654\t131.656\r\n1600\t9.5\t0.0254\t71.3\t0.00420654\t133.756\r\n2000\t9.5\t0.0254\t71.3\t0.00420654\t134.976\r\n2500\t9.5\t0.0254\t71.3\t0.00420654\t135.956\r\n3150\t9.5\t0.0254\t71.3\t0.00420654\t136.166\r\n4000\t9.5\t0.0254\t71.3\t0.00420654\t134.236\r\n5000\t9.5\t0.0254\t71.3\t0.00420654\t131.186\r\n6300\t9.5\t0.0254\t71.3\t0.00420654\t127.246\r\n400\t9.5\t0.0254\t55.5\t0.0043284\t120.952\r\n500\t9.5\t0.0254\t55.5\t0.0043284\t123.082\r\n630\t9.5\t0.0254\t55.5\t0.0043284\t125.452\r\n800\t9.5\t0.0254\t55.5\t0.0043284\t128.082\r\n1000\t9.5\t0.0254\t55.5\t0.0043284\t130.332\r\n1250\t9.5\t0.0254\t55.5\t0.0043284\t132.202\r\n1600\t9.5\t0.0254\t55.5\t0.0043284\t133.062\r\n2000\t9.5\t0.0254\t55.5\t0.0043284\t134.052\r\n2500\t9.5\t0.0254\t55.5\t0.0043284\t134.152\r\n3150\t9.5\t0.0254\t55.5\t0.0043284\t133.252\r\n4000\t9.5\t0.0254\t55.5\t0.0043284\t131.582\r\n5000\t9.5\t0.0254\t55.5\t0.0043284\t128.412\r\n6300\t9.5\t0.0254\t55.5\t0.0043284\t124.222\r\n200\t9.5\t0.0254\t39.6\t0.00449821\t116.074\r\n250\t9.5\t0.0254\t39.6\t0.00449821\t116.924\r\n315\t9.5\t0.0254\t39.6\t0.00449821\t119.294\r\n400\t9.5\t0.0254\t39.6\t0.00449821\t121.154\r\n500\t9.5\t0.0254\t39.6\t0.00449821\t123.894\r\n630\t9.5\t0.0254\t39.6\t0.00449821\t126.514\r\n800\t9.5\t0.0254\t39.6\t0.00449821\t129.014\r\n1000\t9.5\t0.0254\t39.6\t0.00449821\t130.374\r\n1250\t9.5\t0.0254\t39.6\t0.00449821\t130.964\r\n1600\t9.5\t0.0254\t39.6\t0.00449821\t131.184\r\n2000\t9.5\t0.0254\t39.6\t0.00449821\t131.274\r\n2500\t9.5\t0.0254\t39.6\t0.00449821\t131.234\r\n3150\t9.5\t0.0254\t39.6\t0.00449821\t129.934\r\n4000\t9.5\t0.0254\t39.6\t0.00449821\t127.864\r\n5000\t9.5\t0.0254\t39.6\t0.00449821\t125.044\r\n6300\t9.5\t0.0254\t39.6\t0.00449821\t120.324\r\n200\t9.5\t0.0254\t31.7\t0.00461377\t119.146\r\n250\t9.5\t0.0254\t31.7\t0.00461377\t120.136\r\n315\t9.5\t0.0254\t31.7\t0.00461377\t122.766\r\n400\t9.5\t0.0254\t31.7\t0.00461377\t124.756\r\n500\t9.5\t0.0254\t31.7\t0.00461377\t126.886\r\n630\t9.5\t0.0254\t31.7\t0.00461377\t129.006\r\n800\t9.5\t0.0254\t31.7\t0.00461377\t130.746\r\n1000\t9.5\t0.0254\t31.7\t0.00461377\t131.346\r\n1250\t9.5\t0.0254\t31.7\t0.00461377\t131.446\r\n1600\t9.5\t0.0254\t31.7\t0.00461377\t131.036\r\n2000\t9.5\t0.0254\t31.7\t0.00461377\t130.496\r\n2500\t9.5\t0.0254\t31.7\t0.00461377\t130.086\r\n3150\t9.5\t0.0254\t31.7\t0.00461377\t128.536\r\n4000\t9.5\t0.0254\t31.7\t0.00461377\t126.736\r\n5000\t9.5\t0.0254\t31.7\t0.00461377\t124.426\r\n6300\t9.5\t0.0254\t31.7\t0.00461377\t120.726\r\n250\t12.7\t0.0254\t71.3\t0.0121808\t119.698\r\n315\t12.7\t0.0254\t71.3\t0.0121808\t122.938\r\n400\t12.7\t0.0254\t71.3\t0.0121808\t125.048\r\n500\t12.7\t0.0254\t71.3\t0.0121808\t126.898\r\n630\t12.7\t0.0254\t71.3\t0.0121808\t128.878\r\n800\t12.7\t0.0254\t71.3\t0.0121808\t130.348\r\n1000\t12.7\t0.0254\t71.3\t0.0121808\t131.698\r\n1250\t12.7\t0.0254\t71.3\t0.0121808\t133.048\r\n1600\t12.7\t0.0254\t71.3\t0.0121808\t134.528\r\n2000\t12.7\t0.0254\t71.3\t0.0121808\t134.228\r\n2500\t12.7\t0.0254\t71.3\t0.0121808\t134.058\r\n3150\t12.7\t0.0254\t71.3\t0.0121808\t133.758\r\n4000\t12.7\t0.0254\t71.3\t0.0121808\t131.808\r\n5000\t12.7\t0.0254\t71.3\t0.0121808\t128.978\r\n6300\t12.7\t0.0254\t71.3\t0.0121808\t125.398\r\n8000\t12.7\t0.0254\t71.3\t0.0121808\t120.538\r\n10000\t12.7\t0.0254\t71.3\t0.0121808\t114.418\r\n250\t12.7\t0.0254\t39.6\t0.0130253\t121.547\r\n315\t12.7\t0.0254\t39.6\t0.0130253\t123.537\r\n400\t12.7\t0.0254\t39.6\t0.0130253\t125.527\r\n500\t12.7\t0.0254\t39.6\t0.0130253\t127.127\r\n630\t12.7\t0.0254\t39.6\t0.0130253\t128.867\r\n800\t12.7\t0.0254\t39.6\t0.0130253\t130.217\r\n1000\t12.7\t0.0254\t39.6\t0.0130253\t130.947\r\n1250\t12.7\t0.0254\t39.6\t0.0130253\t130.777\r\n1600\t12.7\t0.0254\t39.6\t0.0130253\t129.977\r\n2000\t12.7\t0.0254\t39.6\t0.0130253\t129.567\r\n2500\t12.7\t0.0254\t39.6\t0.0130253\t129.027\r\n3150\t12.7\t0.0254\t39.6\t0.0130253\t127.847\r\n4000\t12.7\t0.0254\t39.6\t0.0130253\t126.537\r\n5000\t12.7\t0.0254\t39.6\t0.0130253\t125.107\r\n6300\t12.7\t0.0254\t39.6\t0.0130253\t123.177\r\n8000\t12.7\t0.0254\t39.6\t0.0130253\t120.607\r\n10000\t12.7\t0.0254\t39.6\t0.0130253\t116.017\r\n200\t17.4\t0.0254\t71.3\t0.016104\t112.506\r\n250\t17.4\t0.0254\t71.3\t0.016104\t113.796\r\n315\t17.4\t0.0254\t71.3\t0.016104\t115.846\r\n400\t17.4\t0.0254\t71.3\t0.016104\t117.396\r\n500\t17.4\t0.0254\t71.3\t0.016104\t119.806\r\n630\t17.4\t0.0254\t71.3\t0.016104\t122.606\r\n800\t17.4\t0.0254\t71.3\t0.016104\t124.276\r\n1000\t17.4\t0.0254\t71.3\t0.016104\t125.816\r\n1250\t17.4\t0.0254\t71.3\t0.016104\t126.356\r\n1600\t17.4\t0.0254\t71.3\t0.016104\t126.406\r\n2000\t17.4\t0.0254\t71.3\t0.016104\t126.826\r\n2500\t17.4\t0.0254\t71.3\t0.016104\t126.746\r\n3150\t17.4\t0.0254\t71.3\t0.016104\t126.536\r\n4000\t17.4\t0.0254\t71.3\t0.016104\t125.586\r\n5000\t17.4\t0.0254\t71.3\t0.016104\t123.126\r\n6300\t17.4\t0.0254\t71.3\t0.016104\t119.916\r\n8000\t17.4\t0.0254\t71.3\t0.016104\t115.466\r\n200\t17.4\t0.0254\t55.5\t0.0165706\t109.951\r\n250\t17.4\t0.0254\t55.5\t0.0165706\t110.491\r\n315\t17.4\t0.0254\t55.5\t0.0165706\t111.911\r\n400\t17.4\t0.0254\t55.5\t0.0165706\t115.461\r\n500\t17.4\t0.0254\t55.5\t0.0165706\t119.621\r\n630\t17.4\t0.0254\t55.5\t0.0165706\t122.411\r\n800\t17.4\t0.0254\t55.5\t0.0165706\t123.091\r\n1000\t17.4\t0.0254\t55.5\t0.0165706\t126.001\r\n1250\t17.4\t0.0254\t55.5\t0.0165706\t129.301\r\n1600\t17.4\t0.0254\t55.5\t0.0165706\t126.471\r\n2000\t17.4\t0.0254\t55.5\t0.0165706\t125.261\r\n2500\t17.4\t0.0254\t55.5\t0.0165706\t124.931\r\n3150\t17.4\t0.0254\t55.5\t0.0165706\t124.101\r\n4000\t17.4\t0.0254\t55.5\t0.0165706\t121.771\r\n5000\t17.4\t0.0254\t55.5\t0.0165706\t118.941\r\n6300\t17.4\t0.0254\t55.5\t0.0165706\t114.861\r\n200\t17.4\t0.0254\t39.6\t0.0172206\t114.044\r\n250\t17.4\t0.0254\t39.6\t0.0172206\t114.714\r\n315\t17.4\t0.0254\t39.6\t0.0172206\t115.144\r\n400\t17.4\t0.0254\t39.6\t0.0172206\t115.444\r\n500\t17.4\t0.0254\t39.6\t0.0172206\t117.514\r\n630\t17.4\t0.0254\t39.6\t0.0172206\t124.514\r\n800\t17.4\t0.0254\t39.6\t0.0172206\t135.324\r\n1000\t17.4\t0.0254\t39.6\t0.0172206\t138.274\r\n1250\t17.4\t0.0254\t39.6\t0.0172206\t131.364\r\n1600\t17.4\t0.0254\t39.6\t0.0172206\t127.614\r\n2000\t17.4\t0.0254\t39.6\t0.0172206\t126.644\r\n2500\t17.4\t0.0254\t39.6\t0.0172206\t124.154\r\n3150\t17.4\t0.0254\t39.6\t0.0172206\t123.564\r\n4000\t17.4\t0.0254\t39.6\t0.0172206\t122.724\r\n5000\t17.4\t0.0254\t39.6\t0.0172206\t119.854\r\n200\t17.4\t0.0254\t31.7\t0.0176631\t116.146\r\n250\t17.4\t0.0254\t31.7\t0.0176631\t116.956\r\n315\t17.4\t0.0254\t31.7\t0.0176631\t118.416\r\n400\t17.4\t0.0254\t31.7\t0.0176631\t120.766\r\n500\t17.4\t0.0254\t31.7\t0.0176631\t127.676\r\n630\t17.4\t0.0254\t31.7\t0.0176631\t136.886\r\n800\t17.4\t0.0254\t31.7\t0.0176631\t139.226\r\n1000\t17.4\t0.0254\t31.7\t0.0176631\t131.796\r\n1250\t17.4\t0.0254\t31.7\t0.0176631\t128.306\r\n1600\t17.4\t0.0254\t31.7\t0.0176631\t126.846\r\n2000\t17.4\t0.0254\t31.7\t0.0176631\t124.356\r\n2500\t17.4\t0.0254\t31.7\t0.0176631\t124.166\r\n3150\t17.4\t0.0254\t31.7\t0.0176631\t123.466\r\n4000\t17.4\t0.0254\t31.7\t0.0176631\t121.996\r\n5000\t17.4\t0.0254\t31.7\t0.0176631\t117.996\r\n315\t22.2\t0.0254\t71.3\t0.0214178\t115.857\r\n400\t22.2\t0.0254\t71.3\t0.0214178\t117.927\r\n500\t22.2\t0.0254\t71.3\t0.0214178\t117.967\r\n630\t22.2\t0.0254\t71.3\t0.0214178\t120.657\r\n800\t22.2\t0.0254\t71.3\t0.0214178\t123.227\r\n1000\t22.2\t0.0254\t71.3\t0.0214178\t134.247\r\n1250\t22.2\t0.0254\t71.3\t0.0214178\t140.987\r\n1600\t22.2\t0.0254\t71.3\t0.0214178\t131.817\r\n2000\t22.2\t0.0254\t71.3\t0.0214178\t127.197\r\n2500\t22.2\t0.0254\t71.3\t0.0214178\t126.097\r\n3150\t22.2\t0.0254\t71.3\t0.0214178\t124.127\r\n4000\t22.2\t0.0254\t71.3\t0.0214178\t123.917\r\n5000\t22.2\t0.0254\t71.3\t0.0214178\t125.727\r\n6300\t22.2\t0.0254\t71.3\t0.0214178\t123.127\r\n8000\t22.2\t0.0254\t71.3\t0.0214178\t121.657\r\n200\t22.2\t0.0254\t39.6\t0.0229028\t116.066\r\n250\t22.2\t0.0254\t39.6\t0.0229028\t117.386\r\n315\t22.2\t0.0254\t39.6\t0.0229028\t120.716\r\n400\t22.2\t0.0254\t39.6\t0.0229028\t123.416\r\n500\t22.2\t0.0254\t39.6\t0.0229028\t129.776\r\n630\t22.2\t0.0254\t39.6\t0.0229028\t137.026\r\n800\t22.2\t0.0254\t39.6\t0.0229028\t137.076\r\n1000\t22.2\t0.0254\t39.6\t0.0229028\t128.416\r\n1250\t22.2\t0.0254\t39.6\t0.0229028\t126.446\r\n1600\t22.2\t0.0254\t39.6\t0.0229028\t122.216\r\n2000\t22.2\t0.0254\t39.6\t0.0229028\t121.256\r\n2500\t22.2\t0.0254\t39.6\t0.0229028\t121.306\r\n3150\t22.2\t0.0254\t39.6\t0.0229028\t120.856\r\n4000\t22.2\t0.0254\t39.6\t0.0229028\t119.646\r\n5000\t22.2\t0.0254\t39.6\t0.0229028\t118.816\r\n630\t0\t0.1016\t71.3\t0.00121072\t124.155\r\n800\t0\t0.1016\t71.3\t0.00121072\t126.805\r\n1000\t0\t0.1016\t71.3\t0.00121072\t128.825\r\n1250\t0\t0.1016\t71.3\t0.00121072\t130.335\r\n1600\t0\t0.1016\t71.3\t0.00121072\t131.725\r\n2000\t0\t0.1016\t71.3\t0.00121072\t132.095\r\n2500\t0\t0.1016\t71.3\t0.00121072\t132.595\r\n3150\t0\t0.1016\t71.3\t0.00121072\t131.955\r\n4000\t0\t0.1016\t71.3\t0.00121072\t130.935\r\n5000\t0\t0.1016\t71.3\t0.00121072\t130.795\r\n6300\t0\t0.1016\t71.3\t0.00121072\t129.395\r\n8000\t0\t0.1016\t71.3\t0.00121072\t125.465\r\n10000\t0\t0.1016\t71.3\t0.00121072\t123.305\r\n12500\t0\t0.1016\t71.3\t0.00121072\t119.375\r\n630\t0\t0.1016\t55.5\t0.00131983\t126.170\r\n800\t0\t0.1016\t55.5\t0.00131983\t127.920\r\n1000\t0\t0.1016\t55.5\t0.00131983\t129.800\r\n1250\t0\t0.1016\t55.5\t0.00131983\t131.430\r\n1600\t0\t0.1016\t55.5\t0.00131983\t132.050\r\n2000\t0\t0.1016\t55.5\t0.00131983\t132.540\r\n2500\t0\t0.1016\t55.5\t0.00131983\t133.040\r\n3150\t0\t0.1016\t55.5\t0.00131983\t131.780\r\n4000\t0\t0.1016\t55.5\t0.00131983\t129.500\r\n5000\t0\t0.1016\t55.5\t0.00131983\t128.360\r\n6300\t0\t0.1016\t55.5\t0.00131983\t127.730\r\n8000\t0\t0.1016\t55.5\t0.00131983\t124.450\r\n10000\t0\t0.1016\t55.5\t0.00131983\t121.930\r\n12500\t0\t0.1016\t55.5\t0.00131983\t119.910\r\n630\t0\t0.1016\t39.6\t0.00146332\t125.401\r\n800\t0\t0.1016\t39.6\t0.00146332\t128.401\r\n1000\t0\t0.1016\t39.6\t0.00146332\t130.781\r\n1250\t0\t0.1016\t39.6\t0.00146332\t132.271\r\n1600\t0\t0.1016\t39.6\t0.00146332\t133.261\r\n2000\t0\t0.1016\t39.6\t0.00146332\t133.251\r\n2500\t0\t0.1016\t39.6\t0.00146332\t132.611\r\n3150\t0\t0.1016\t39.6\t0.00146332\t130.961\r\n4000\t0\t0.1016\t39.6\t0.00146332\t127.801\r\n5000\t0\t0.1016\t39.6\t0.00146332\t126.021\r\n6300\t0\t0.1016\t39.6\t0.00146332\t125.631\r\n8000\t0\t0.1016\t39.6\t0.00146332\t122.341\r\n10000\t0\t0.1016\t39.6\t0.00146332\t119.561\r\n630\t0\t0.1016\t31.7\t0.00150092\t126.413\r\n800\t0\t0.1016\t31.7\t0.00150092\t129.053\r\n1000\t0\t0.1016\t31.7\t0.00150092\t131.313\r\n1250\t0\t0.1016\t31.7\t0.00150092\t133.063\r\n1600\t0\t0.1016\t31.7\t0.00150092\t133.553\r\n2000\t0\t0.1016\t31.7\t0.00150092\t133.153\r\n2500\t0\t0.1016\t31.7\t0.00150092\t132.003\r\n3150\t0\t0.1016\t31.7\t0.00150092\t129.973\r\n4000\t0\t0.1016\t31.7\t0.00150092\t126.933\r\n5000\t0\t0.1016\t31.7\t0.00150092\t124.393\r\n6300\t0\t0.1016\t31.7\t0.00150092\t124.253\r\n8000\t0\t0.1016\t31.7\t0.00150092\t120.193\r\n10000\t0\t0.1016\t31.7\t0.00150092\t115.893\r\n800\t3.3\t0.1016\t71.3\t0.00202822\t131.074\r\n1000\t3.3\t0.1016\t71.3\t0.00202822\t131.434\r\n1250\t3.3\t0.1016\t71.3\t0.00202822\t132.304\r\n1600\t3.3\t0.1016\t71.3\t0.00202822\t133.664\r\n2000\t3.3\t0.1016\t71.3\t0.00202822\t134.034\r\n2500\t3.3\t0.1016\t71.3\t0.00202822\t133.894\r\n3150\t3.3\t0.1016\t71.3\t0.00202822\t132.114\r\n4000\t3.3\t0.1016\t71.3\t0.00202822\t128.704\r\n5000\t3.3\t0.1016\t71.3\t0.00202822\t127.054\r\n6300\t3.3\t0.1016\t71.3\t0.00202822\t124.904\r\n8000\t3.3\t0.1016\t71.3\t0.00202822\t121.234\r\n10000\t3.3\t0.1016\t71.3\t0.00202822\t116.694\r\n630\t3.3\t0.1016\t55.5\t0.002211\t126.599\r\n800\t3.3\t0.1016\t55.5\t0.002211\t129.119\r\n1000\t3.3\t0.1016\t55.5\t0.002211\t131.129\r\n1250\t3.3\t0.1016\t55.5\t0.002211\t132.769\r\n1600\t3.3\t0.1016\t55.5\t0.002211\t133.649\r\n2000\t3.3\t0.1016\t55.5\t0.002211\t133.649\r\n2500\t3.3\t0.1016\t55.5\t0.002211\t132.889\r\n3150\t3.3\t0.1016\t55.5\t0.002211\t130.629\r\n4000\t3.3\t0.1016\t55.5\t0.002211\t127.229\r\n5000\t3.3\t0.1016\t55.5\t0.002211\t124.839\r\n6300\t3.3\t0.1016\t55.5\t0.002211\t123.839\r\n8000\t3.3\t0.1016\t55.5\t0.002211\t120.569\r\n10000\t3.3\t0.1016\t55.5\t0.002211\t115.659\r\n630\t3.3\t0.1016\t39.6\t0.00245138\t127.251\r\n800\t3.3\t0.1016\t39.6\t0.00245138\t129.991\r\n1000\t3.3\t0.1016\t39.6\t0.00245138\t131.971\r\n1250\t3.3\t0.1016\t39.6\t0.00245138\t133.211\r\n1600\t3.3\t0.1016\t39.6\t0.00245138\t133.071\r\n2000\t3.3\t0.1016\t39.6\t0.00245138\t132.301\r\n2500\t3.3\t0.1016\t39.6\t0.00245138\t130.791\r\n3150\t3.3\t0.1016\t39.6\t0.00245138\t128.401\r\n4000\t3.3\t0.1016\t39.6\t0.00245138\t124.881\r\n5000\t3.3\t0.1016\t39.6\t0.00245138\t122.371\r\n6300\t3.3\t0.1016\t39.6\t0.00245138\t120.851\r\n8000\t3.3\t0.1016\t39.6\t0.00245138\t118.091\r\n10000\t3.3\t0.1016\t39.6\t0.00245138\t115.321\r\n630\t3.3\t0.1016\t31.7\t0.00251435\t128.952\r\n800\t3.3\t0.1016\t31.7\t0.00251435\t131.362\r\n1000\t3.3\t0.1016\t31.7\t0.00251435\t133.012\r\n1250\t3.3\t0.1016\t31.7\t0.00251435\t134.022\r\n1600\t3.3\t0.1016\t31.7\t0.00251435\t133.402\r\n2000\t3.3\t0.1016\t31.7\t0.00251435\t131.642\r\n2500\t3.3\t0.1016\t31.7\t0.00251435\t130.392\r\n3150\t3.3\t0.1016\t31.7\t0.00251435\t128.252\r\n4000\t3.3\t0.1016\t31.7\t0.00251435\t124.852\r\n5000\t3.3\t0.1016\t31.7\t0.00251435\t122.082\r\n6300\t3.3\t0.1016\t31.7\t0.00251435\t120.702\r\n8000\t3.3\t0.1016\t31.7\t0.00251435\t117.432\r\n630\t6.7\t0.1016\t71.3\t0.00478288\t131.448\r\n800\t6.7\t0.1016\t71.3\t0.00478288\t134.478\r\n1000\t6.7\t0.1016\t71.3\t0.00478288\t136.758\r\n1250\t6.7\t0.1016\t71.3\t0.00478288\t137.658\r\n1600\t6.7\t0.1016\t71.3\t0.00478288\t136.678\r\n2000\t6.7\t0.1016\t71.3\t0.00478288\t134.568\r\n2500\t6.7\t0.1016\t71.3\t0.00478288\t131.458\r\n3150\t6.7\t0.1016\t71.3\t0.00478288\t124.458\r\n500\t6.7\t0.1016\t55.5\t0.0052139\t129.343\r\n630\t6.7\t0.1016\t55.5\t0.0052139\t133.023\r\n800\t6.7\t0.1016\t55.5\t0.0052139\t135.953\r\n1000\t6.7\t0.1016\t55.5\t0.0052139\t137.233\r\n1250\t6.7\t0.1016\t55.5\t0.0052139\t136.883\r\n1600\t6.7\t0.1016\t55.5\t0.0052139\t133.653\r\n2000\t6.7\t0.1016\t55.5\t0.0052139\t129.653\r\n2500\t6.7\t0.1016\t55.5\t0.0052139\t124.273\r\n400\t6.7\t0.1016\t39.6\t0.00578076\t128.295\r\n500\t6.7\t0.1016\t39.6\t0.00578076\t130.955\r\n630\t6.7\t0.1016\t39.6\t0.00578076\t133.355\r\n800\t6.7\t0.1016\t39.6\t0.00578076\t134.625\r\n1000\t6.7\t0.1016\t39.6\t0.00578076\t134.515\r\n1250\t6.7\t0.1016\t39.6\t0.00578076\t132.395\r\n1600\t6.7\t0.1016\t39.6\t0.00578076\t127.375\r\n2000\t6.7\t0.1016\t39.6\t0.00578076\t122.235\r\n315\t6.7\t0.1016\t31.7\t0.00592927\t126.266\r\n400\t6.7\t0.1016\t31.7\t0.00592927\t128.296\r\n500\t6.7\t0.1016\t31.7\t0.00592927\t130.206\r\n630\t6.7\t0.1016\t31.7\t0.00592927\t132.116\r\n800\t6.7\t0.1016\t31.7\t0.00592927\t132.886\r\n1000\t6.7\t0.1016\t31.7\t0.00592927\t131.636\r\n1250\t6.7\t0.1016\t31.7\t0.00592927\t129.256\r\n1600\t6.7\t0.1016\t31.7\t0.00592927\t124.346\r\n2000\t6.7\t0.1016\t31.7\t0.00592927\t120.446\r\n200\t8.9\t0.1016\t71.3\t0.0103088\t133.503\r\n250\t8.9\t0.1016\t71.3\t0.0103088\t134.533\r\n315\t8.9\t0.1016\t71.3\t0.0103088\t136.583\r\n400\t8.9\t0.1016\t71.3\t0.0103088\t138.123\r\n500\t8.9\t0.1016\t71.3\t0.0103088\t138.523\r\n630\t8.9\t0.1016\t71.3\t0.0103088\t138.423\r\n800\t8.9\t0.1016\t71.3\t0.0103088\t137.813\r\n1000\t8.9\t0.1016\t71.3\t0.0103088\t135.433\r\n1250\t8.9\t0.1016\t71.3\t0.0103088\t132.793\r\n1600\t8.9\t0.1016\t71.3\t0.0103088\t128.763\r\n2000\t8.9\t0.1016\t71.3\t0.0103088\t124.233\r\n2500\t8.9\t0.1016\t71.3\t0.0103088\t123.623\r\n3150\t8.9\t0.1016\t71.3\t0.0103088\t123.263\r\n4000\t8.9\t0.1016\t71.3\t0.0103088\t120.243\r\n5000\t8.9\t0.1016\t71.3\t0.0103088\t116.723\r\n6300\t8.9\t0.1016\t71.3\t0.0103088\t117.253\r\n200\t8.9\t0.1016\t39.6\t0.0124596\t133.420\r\n250\t8.9\t0.1016\t39.6\t0.0124596\t134.340\r\n315\t8.9\t0.1016\t39.6\t0.0124596\t135.380\r\n400\t8.9\t0.1016\t39.6\t0.0124596\t135.540\r\n500\t8.9\t0.1016\t39.6\t0.0124596\t133.790\r\n630\t8.9\t0.1016\t39.6\t0.0124596\t131.920\r\n800\t8.9\t0.1016\t39.6\t0.0124596\t130.940\r\n1000\t8.9\t0.1016\t39.6\t0.0124596\t129.580\r\n1250\t8.9\t0.1016\t39.6\t0.0124596\t127.710\r\n1600\t8.9\t0.1016\t39.6\t0.0124596\t123.820\r\n2000\t8.9\t0.1016\t39.6\t0.0124596\t119.040\r\n2500\t8.9\t0.1016\t39.6\t0.0124596\t119.190\r\n3150\t8.9\t0.1016\t39.6\t0.0124596\t119.350\r\n4000\t8.9\t0.1016\t39.6\t0.0124596\t116.220\r\n5000\t8.9\t0.1016\t39.6\t0.0124596\t113.080\r\n6300\t8.9\t0.1016\t39.6\t0.0124596\t113.110\r\n200\t12.3\t0.1016\t71.3\t0.0337792\t130.588\r\n250\t12.3\t0.1016\t71.3\t0.0337792\t131.568\r\n315\t12.3\t0.1016\t71.3\t0.0337792\t137.068\r\n400\t12.3\t0.1016\t71.3\t0.0337792\t139.428\r\n500\t12.3\t0.1016\t71.3\t0.0337792\t140.158\r\n630\t12.3\t0.1016\t71.3\t0.0337792\t135.368\r\n800\t12.3\t0.1016\t71.3\t0.0337792\t127.318\r\n1000\t12.3\t0.1016\t71.3\t0.0337792\t127.928\r\n1250\t12.3\t0.1016\t71.3\t0.0337792\t126.648\r\n1600\t12.3\t0.1016\t71.3\t0.0337792\t124.748\r\n2000\t12.3\t0.1016\t71.3\t0.0337792\t122.218\r\n2500\t12.3\t0.1016\t71.3\t0.0337792\t121.318\r\n3150\t12.3\t0.1016\t71.3\t0.0337792\t120.798\r\n4000\t12.3\t0.1016\t71.3\t0.0337792\t118.018\r\n5000\t12.3\t0.1016\t71.3\t0.0337792\t116.108\r\n6300\t12.3\t0.1016\t71.3\t0.0337792\t113.958\r\n200\t12.3\t0.1016\t55.5\t0.0368233\t132.304\r\n250\t12.3\t0.1016\t55.5\t0.0368233\t133.294\r\n315\t12.3\t0.1016\t55.5\t0.0368233\t135.674\r\n400\t12.3\t0.1016\t55.5\t0.0368233\t136.414\r\n500\t12.3\t0.1016\t55.5\t0.0368233\t133.774\r\n630\t12.3\t0.1016\t55.5\t0.0368233\t124.244\r\n800\t12.3\t0.1016\t55.5\t0.0368233\t125.114\r\n1000\t12.3\t0.1016\t55.5\t0.0368233\t125.484\r\n1250\t12.3\t0.1016\t55.5\t0.0368233\t124.214\r\n1600\t12.3\t0.1016\t55.5\t0.0368233\t121.824\r\n2000\t12.3\t0.1016\t55.5\t0.0368233\t118.564\r\n2500\t12.3\t0.1016\t55.5\t0.0368233\t117.054\r\n3150\t12.3\t0.1016\t55.5\t0.0368233\t116.914\r\n4000\t12.3\t0.1016\t55.5\t0.0368233\t114.404\r\n5000\t12.3\t0.1016\t55.5\t0.0368233\t112.014\r\n6300\t12.3\t0.1016\t55.5\t0.0368233\t110.124\r\n200\t12.3\t0.1016\t39.6\t0.0408268\t128.545\r\n250\t12.3\t0.1016\t39.6\t0.0408268\t129.675\r\n315\t12.3\t0.1016\t39.6\t0.0408268\t129.415\r\n400\t12.3\t0.1016\t39.6\t0.0408268\t128.265\r\n500\t12.3\t0.1016\t39.6\t0.0408268\t122.205\r\n630\t12.3\t0.1016\t39.6\t0.0408268\t121.315\r\n800\t12.3\t0.1016\t39.6\t0.0408268\t122.315\r\n1000\t12.3\t0.1016\t39.6\t0.0408268\t122.435\r\n1250\t12.3\t0.1016\t39.6\t0.0408268\t121.165\r\n1600\t12.3\t0.1016\t39.6\t0.0408268\t117.875\r\n2000\t12.3\t0.1016\t39.6\t0.0408268\t114.085\r\n2500\t12.3\t0.1016\t39.6\t0.0408268\t113.315\r\n3150\t12.3\t0.1016\t39.6\t0.0408268\t113.055\r\n4000\t12.3\t0.1016\t39.6\t0.0408268\t110.905\r\n5000\t12.3\t0.1016\t39.6\t0.0408268\t108.625\r\n6300\t12.3\t0.1016\t39.6\t0.0408268\t107.985\r\n200\t12.3\t0.1016\t31.7\t0.0418756\t124.987\r\n250\t12.3\t0.1016\t31.7\t0.0418756\t125.857\r\n315\t12.3\t0.1016\t31.7\t0.0418756\t124.717\r\n400\t12.3\t0.1016\t31.7\t0.0418756\t123.207\r\n500\t12.3\t0.1016\t31.7\t0.0418756\t118.667\r\n630\t12.3\t0.1016\t31.7\t0.0418756\t119.287\r\n800\t12.3\t0.1016\t31.7\t0.0418756\t120.037\r\n1000\t12.3\t0.1016\t31.7\t0.0418756\t119.777\r\n1250\t12.3\t0.1016\t31.7\t0.0418756\t118.767\r\n1600\t12.3\t0.1016\t31.7\t0.0418756\t114.477\r\n2000\t12.3\t0.1016\t31.7\t0.0418756\t110.447\r\n2500\t12.3\t0.1016\t31.7\t0.0418756\t110.317\r\n3150\t12.3\t0.1016\t31.7\t0.0418756\t110.307\r\n4000\t12.3\t0.1016\t31.7\t0.0418756\t108.407\r\n5000\t12.3\t0.1016\t31.7\t0.0418756\t107.147\r\n6300\t12.3\t0.1016\t31.7\t0.0418756\t107.267\r\n200\t15.6\t0.1016\t71.3\t0.0437259\t130.898\r\n250\t15.6\t0.1016\t71.3\t0.0437259\t132.158\r\n315\t15.6\t0.1016\t71.3\t0.0437259\t133.808\r\n400\t15.6\t0.1016\t71.3\t0.0437259\t134.058\r\n500\t15.6\t0.1016\t71.3\t0.0437259\t130.638\r\n630\t15.6\t0.1016\t71.3\t0.0437259\t122.288\r\n800\t15.6\t0.1016\t71.3\t0.0437259\t124.188\r\n1000\t15.6\t0.1016\t71.3\t0.0437259\t124.438\r\n1250\t15.6\t0.1016\t71.3\t0.0437259\t123.178\r\n1600\t15.6\t0.1016\t71.3\t0.0437259\t121.528\r\n2000\t15.6\t0.1016\t71.3\t0.0437259\t119.888\r\n2500\t15.6\t0.1016\t71.3\t0.0437259\t118.998\r\n3150\t15.6\t0.1016\t71.3\t0.0437259\t116.468\r\n4000\t15.6\t0.1016\t71.3\t0.0437259\t113.298\r\n200\t15.6\t0.1016\t39.6\t0.0528487\t123.514\r\n250\t15.6\t0.1016\t39.6\t0.0528487\t124.644\r\n315\t15.6\t0.1016\t39.6\t0.0528487\t122.754\r\n400\t15.6\t0.1016\t39.6\t0.0528487\t120.484\r\n500\t15.6\t0.1016\t39.6\t0.0528487\t115.304\r\n630\t15.6\t0.1016\t39.6\t0.0528487\t118.084\r\n800\t15.6\t0.1016\t39.6\t0.0528487\t118.964\r\n1000\t15.6\t0.1016\t39.6\t0.0528487\t119.224\r\n1250\t15.6\t0.1016\t39.6\t0.0528487\t118.214\r\n1600\t15.6\t0.1016\t39.6\t0.0528487\t114.554\r\n2000\t15.6\t0.1016\t39.6\t0.0528487\t110.894\r\n2500\t15.6\t0.1016\t39.6\t0.0528487\t110.264\r\n3150\t15.6\t0.1016\t39.6\t0.0528487\t109.254\r\n4000\t15.6\t0.1016\t39.6\t0.0528487\t106.604\r\n5000\t15.6\t0.1016\t39.6\t0.0528487\t106.224\r\n6300\t15.6\t0.1016\t39.6\t0.0528487\t104.204\r\n'

columnas = ["f_Hz", "alpha_deg", "c_m", "U_ms", "delta_m", "L_dB"]
df = pd.read_csv(StringIO(DATOS_UCI), sep=r"\s+", names=columnas)
assert df.shape == (1503, 6)
assert np.isfinite(df.to_numpy()).all()
assert (df[["f_Hz", "c_m", "U_ms", "delta_m"]] > 0).all().all()
print("Filas y columnas:", df.shape)
print("Filas duplicadas:", df.duplicated().sum())
display(df.head())


### 2.1. Leer una observación
`X` reunirá las cinco entradas; `y`, el nivel sonoro. Antes de construirlos, interpreta una fila con sus unidades. Por ejemplo, la frecuencia indica el punto del espectro en el que se midió el nivel.


In [ ]:
fila = df.iloc[0]
for nombre, valor in fila.items():
    print(f'{nombre}: {valor:g}')
display(df.describe().round(3))
print('Velocidades disponibles [m/s]:', np.sort(df.U_ms.unique()))


### 2.2. Reconocer las escalas
Los histogramas resumen la cobertura de cada magnitud. Observa los rangos: frecuencia y espesor tienen escalas numéricas muy diferentes. Esta diferencia motivará la estandarización.

Esta exploración inicial describe el conjunto completo. Las decisiones de ajuste y selección que siguen utilizarán entrenamiento y validación; las métricas predictivas de prueba se calculan al final.


In [ ]:
df.hist(bins=25, figsize=(11, 7))
plt.tight_layout()
plt.show()


### 2.3. Dibujar un espectro
Agrupamos las filas por ángulo, cuerda, velocidad y espesor. Elegimos el grupo con más frecuencias y ordenamos sus puntos. El eje logarítmico permite comparar intervalos de frecuencia por razones, por ejemplo 1.000–2.000 y 2.000–4.000 Hz.


In [ ]:
condiciones = ['alpha_deg', 'c_m', 'U_ms', 'delta_m']
grupos = df.groupby(condiciones)
clave = grupos.size().idxmax()
espectro = grupos.get_group(clave).sort_values('f_Hz')
print('Condiciones:', dict(zip(condiciones, clave)))
print('Puntos del espectro:', len(espectro))
plt.semilogx(espectro.f_Hz, espectro.L_dB, 'o-')
plt.xlabel('Frecuencia [Hz]')
plt.ylabel('Nivel sonoro escalado [dB]')
plt.title('Espectro a condiciones constantes')
plt.show()


### Actividad 2. Leer los datos

Describe la primera fila con sus unidades. Luego indica el rango de frecuencias y localiza el máximo del espectro mostrado. Explica qué aporta mantener constantes las otras cuatro entradas.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 3. Separar los usos de los datos
**Entrenamiento:** ajusta pesos y sesgos. **Validación:** permite elegir arquitectura, tasa de aprendizaje y época. **Prueba (test):** estima el desempeño del modelo elegido.

Usaremos dos protocolos, cada uno con su propio modelo:

**A. Aleatorio.** Aproximadamente 70 % para entrenar, 15 % para validar y 15 % para probar. Responde cómo se predicen filas reservadas de una distribución semejante. Puntos de un mismo espectro pueden aparecer en distintos subconjuntos.

**B. Velocidad reservada.** Las velocidades menores forman entrenamiento, la segunda más alta forma validación y la máxima forma prueba. Responde cómo se transfiere el modelo hacia velocidades superiores. Otras condiciones también pueden cambiar entre grupos, por lo que la interpretación incluye el cambio conjunto de las entradas.

La celda crea índices de filas. Los usaremos para seleccionar exactamente los mismos ejemplos en cada comparación.


In [ ]:
indices = np.arange(len(df))
train_a, resto = train_test_split(indices, test_size=0.30, random_state=SEED)
val_a, test_a = train_test_split(resto, test_size=0.50, random_state=SEED)
velocidades = np.sort(df.U_ms.unique())
splits = {
    'Aleatoria': (train_a, val_a, test_a),
    'Velocidad reservada': (
        indices[df.U_ms.to_numpy() < velocidades[-2]],
        indices[df.U_ms.to_numpy() == velocidades[-2]],
        indices[df.U_ms.to_numpy() == velocidades[-1]])}
for nombre, partes in splits.items():
    assert len(np.unique(np.concatenate(partes))) == len(df)
    assert sum(map(len, partes)) == len(df)
    print('\n' + nombre)
    for etiqueta, ix in zip(['Entrenamiento', 'Validación', 'Prueba'], partes):
        print(etiqueta, len(ix), 'filas; velocidades:', np.sort(df.iloc[ix].U_ms.unique()))


### Actividad 3. Explicar qué se evalúa

Calcula los porcentajes de cada subconjunto. Identifica las velocidades de validación y prueba del protocolo B. Explica qué pregunta responde cada protocolo y cuál se parece más a predecir una condición nueva de operación.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 4. Preparar entradas y salida
### 4.1. Formas de los arreglos
En scikit-learn, cada fila es una observación: $X$ tiene forma $(N,5)$ y $y$ forma $(N,)$. En una notación con observaciones por columnas, esta matriz corresponde a la transpuesta de $A^{[0]}$.

### 4.2. Estandarizar
Para cada entrada calculamos la media y desviación de entrenamiento:
$$z_j=\frac{x_j-\mu_{j,\mathrm{train}}}{\sigma_{j,\mathrm{train}}}.$$
La misma transformación se aplica a validación y prueba. También estandarizamos $L$. Para recuperar la predicción física usamos
$$\hat L=\mu_{L,\mathrm{train}}+\sigma_{L,\mathrm{train}}\hat z_L.$$
Así, el entrenamiento trabaja con escalas comparables y la evaluación se expresa en dB. `fit` aprende las estadísticas; `transform` las aplica. Aprenderlas con entrenamiento mantiene el papel independiente de los datos reservados.


In [ ]:
entradas = ['f_Hz', 'alpha_deg', 'c_m', 'U_ms', 'delta_m']

def make_features(frame, kind='original'):
    if kind == 'original':
        return frame[entradas].to_numpy(dtype=float)
    if kind == 'dimensional':
        return np.column_stack([
            np.log10(frame.f_Hz * frame.c_m / frame.U_ms),
            np.deg2rad(frame.alpha_deg),
            np.log10(frame.delta_m / frame.c_m),
            np.log10(frame.U_ms / 1.0)])  # Referencia: 1 m/s
    raise ValueError('Elige original o dimensional')

def prepare(partes, kind='original'):
    X = make_features(df, kind)
    y = df.L_dB.to_numpy()
    it, iv, ie = partes
    sx = StandardScaler().fit(X[it])
    sy = StandardScaler().fit(y[it, None])
    xs = [sx.transform(X[ix]) for ix in partes]
    ys = [sy.transform(y[ix, None]).ravel() for ix in partes]
    return xs, ys, sx, sy

xs, ys, sx, sy = prepare(splits['Aleatoria'])
for etiqueta, X, y in zip(['train', 'valid', 'test'], xs, ys):
    print(etiqueta, 'X:', X.shape, 'y:', y.shape)
print('Medias de X_train escalado:', xs[0].mean(axis=0).round(6))
print('Desviaciones:', xs[0].std(axis=0).round(6))


### Actividad 4. Seguir una transformación

Usa las estadísticas de `sy` para transformar un nivel de $120$ dB y luego recuperar su valor original. Explica qué significa una salida estandarizada negativa y qué conjunto determina las estadísticas.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 5. Construir referencias de comparación
Primero tendremos dos referencias: predecir siempre la media de entrenamiento y ajustar una regresión lineal. La primera representa una predicción constante; la segunda combina las cinco entradas con pesos y un sesgo.

Mediremos el desempeño con
$$\mathrm{MAE}=\frac{1}{n}\sum_i |\hat L_i-L_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(\hat L_i-L_i)^2}.$$
Ambas se expresan en dB. RMSE da más peso a errores grandes. También calculamos $R^2=1-\sum_i(L_i-\hat L_i)^2/\sum_i(L_i-\bar L)^2$, donde $\bar L$ es la media del subconjunto evaluado. Un $R^2$ negativo indica que esa media describiría mejor el subconjunto en error cuadrático.

**Ejemplo:** errores de 1 y 3 dB producen MAE = 2 dB y RMSE = $\sqrt{5}\approx2.24$ dB.


In [ ]:
def metrics(y, pred):
    return {'RMSE_dB': np.sqrt(mean_squared_error(y, pred)),
            'MAE_dB': mean_absolute_error(y, pred),
            'R2': r2_score(y, pred)}

def predict_db(model, X, sy):
    pred_z = model.predict(X)
    return sy.inverse_transform(pred_z.reshape(-1, 1)).ravel()

lineal_ejemplo = LinearRegression().fit(xs[0], ys[0])
y_valid = df.iloc[splits['Aleatoria'][1]].L_dB.to_numpy()
pred_valid = predict_db(lineal_ejemplo, xs[1], sy)
print('Referencia lineal en validación:', metrics(y_valid, pred_valid))


## 6. Entender y entrenar la red
### 6.1. De una capa a una predicción
Para una observación $\mathbf x$ estandarizada, una capa oculta realiza
$$\mathbf z^{[1]}=\Theta^{[1]}\mathbf x+\mathbf b^{[1]},\qquad
\mathbf a^{[1]}=\mathrm{ReLU}(\mathbf z^{[1]}),$$
con $\mathrm{ReLU}(z)=\max(0,z)$. La salida es lineal:
$$\hat z_L=\Theta^{[2]}\mathbf a^{[1]}+b^{[2]}.$$
La activación permite describir relaciones no lineales. En una red 5 → 16 → 1, la primera capa tiene $5\times16+16=96$ parámetros y la salida $16+1=17$: en total, **113 parámetros**.

`hidden=(16,)` indica una capa oculta de 16 neuronas. `hidden=(32,32)` indica dos capas de 32. La dimensión de entrada se deduce de los datos; la salida es un único número.

### 6.2. Qué ocurre durante una época
1. Se recorren los ejemplos de entrenamiento en lotes de hasta 64 filas.
2. Se calculan predicciones y el error cuadrático en la escala estandarizada.
3. La retropropagación calcula derivadas respecto de pesos y sesgos.
4. Adam utiliza esas derivadas para actualizar los parámetros. `lr` controla la escala del paso.
5. Al terminar la época se evalúan entrenamiento y validación con el mismo estado de la red.

`partial_fit` ejecuta una época y conserva el estado del optimizador entre llamadas. scikit-learn realiza internamente la retropropagación. Su pérdida de regresión es la mitad del MSE; las curvas que calculamos abajo muestran el **MSE completo**.

### 6.3. Parada temprana
Guardamos la red cuando el MSE de validación mejora al menos `1e-6`. Si pasan `patience=40` épocas sin esa mejora, terminamos. Al final recuperamos el mejor estado guardado. El límite de trabajo es `max_epochs=400`.


In [ ]:
def train_network(xs, ys, hidden, lr, seed=SEED, max_epochs=400, patience=40):
    model = MLPRegressor(hidden_layer_sizes=hidden, activation='relu',
                         solver='adam', learning_rate_init=lr, batch_size=64,
                         alpha=0.0, random_state=seed, shuffle=True,
                         early_stopping=False)
    best_loss = np.inf
    best_model = None
    best_epoch = 0
    espera = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.partial_fit(xs[0], ys[0])
        train_loss = mean_squared_error(ys[0], model.predict(xs[0]))
        valid_loss = mean_squared_error(ys[1], model.predict(xs[1]))
        history.append((train_loss, valid_loss))
        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_model = copy.deepcopy(model)
            best_epoch = epoch
            espera = 0
        else:
            espera += 1
        if espera >= patience:
            break
    return best_model, np.array(history), best_epoch


### Actividad 5. Leer la arquitectura y anticipar el aprendizaje

Calcula los parámetros de la arquitectura 5 → 32 → 32 → 1. Explica el papel de `lr`, `max_epochs` y `patience`. Anticipa qué patrón en las curvas te haría elegir una época anterior a la última.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 7. Comparar configuraciones con validación
Probaremos tres configuraciones. Las dos primeras permiten explorar el cambio de arquitectura con la misma tasa; las dos últimas cambian la tasa manteniendo la arquitectura.

Para cada protocolo repetimos: preparar los datos, ajustar la referencia lineal, entrenar las tres redes y elegir el menor RMSE de validación. Guardamos todas las curvas para poder interpretar la selección.


In [ ]:
configs = [
    {'hidden': (16,), 'lr': 1e-3},
    {'hidden': (32, 32), 'lr': 1e-3},
    {'hidden': (32, 32), 'lr': 3e-3},
]

def select_models(partes, kind='original'):
    xs, ys, sx, sy = prepare(partes, kind)
    y_valid = df.iloc[partes[1]].L_dB.to_numpy()
    linear = LinearRegression().fit(xs[0], ys[0])
    trials, rows = [], []
    for config in configs:
        model, history, best_epoch = train_network(xs, ys, **config)
        score = metrics(y_valid, predict_db(model, xs[1], sy))
        trials.append({'model': model, 'history': history, 'best_epoch': best_epoch})
        rows.append({**config, **score, 'epochs': len(history), 'best_epoch': best_epoch})
    table = pd.DataFrame(rows)
    best = int(table.RMSE_dB.to_numpy().argmin())
    return {'xs': xs, 'sy': sy, 'sx': sx, 'linear': linear,
            'net': trials[best]['model'], 'trials': trials,
            'selection': table, 'best_config': configs[best].copy(),
            'parts': partes, 'kind': kind}

runs = {}
for nombre, partes in splits.items():
    print('Entrenando:', nombre)
    runs[nombre] = select_models(partes)
    display(runs[nombre]['selection'].round(4))
    print('Configuración elegida:', runs[nombre]['best_config'])


### 7.1. Interpretar las curvas
Cada panel muestra una configuración. La línea vertical identifica la época recuperada. Compara las curvas dentro de cada protocolo: cada uno usa su propia escala del objetivo, calculada con su entrenamiento.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for fila_ax, (nombre, run) in zip(axes, runs.items()):
    for ax, config, trial in zip(fila_ax, configs, run['trials']):
        h = trial['history']
        epocas = np.arange(1, len(h) + 1)
        ax.plot(epocas, h[:, 0], label='Entrenamiento')
        ax.plot(epocas, h[:, 1], label='Validación')
        ax.axvline(trial['best_epoch'], color='gray', linestyle=':')
        ax.set(title=f"{nombre}\n{config['hidden']}, lr={config['lr']}",
               xlabel='Época', ylabel='MSE estandarizado')
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### Actividad 6. Justificar una selección

Para cada protocolo, registra la configuración elegida, su RMSE de validación, las épocas ejecutadas y la época recuperada. Describe una curva y relaciona su comportamiento con tu hipótesis de la actividad 5.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 8. Extensión opcional: representar las entradas con física
**Momento de realización:** durante el desarrollo, antes de calcular las métricas finales de prueba.

Consideramos el número $St_c=fc/U$ y la razón $\delta^*/c$. Verifica sus unidades: $[fc/U]=(s^{-1}m)/(m\,s^{-1})=1$ y $[\delta^*/c]=m/m=1$.

La representación alternativa contiene cuatro entradas: $\log_{10}(St_c)$, $\alpha$ en radianes, $\log_{10}(\delta^*/c)$ y $\log_{10}(U/U_\mathrm{ref})$, con $U_\mathrm{ref}=1$ m/s. Los argumentos de los logaritmos son adimensionales y positivos. La referencia de velocidad conserva información de escala; una descripción de similitud completa requeriría considerar las variables físicas pertinentes, como las que forman Mach y Reynolds.

Pasamos de cinco entradas a cuatro: esta transformación combina variables y puede perder información independiente. Por ello interesa estudiar tanto una mejora como un empeoramiento.

Para realizar la extensión, cambia `HACER_EXTENSION` a `True`. Seleccionaremos la representación según el mejor RMSE de validación de cada familia. La comparación lineal utilizará las mismas entradas que la red seleccionada.


In [ ]:
HACER_EXTENSION = False
if HACER_EXTENSION:
    for nombre, partes in splits.items():
        alternativa = select_models(partes, kind='dimensional')
        original_rmse = runs[nombre]['selection'].RMSE_dB.min()
        alternativa_rmse = alternativa['selection'].RMSE_dB.min()
        print(nombre, 'RMSE validación original:', original_rmse,
              'dimensional:', alternativa_rmse)
        display(alternativa['selection'].round(4))
        if alternativa_rmse < original_rmse:
            runs[nombre] = alternativa
        print('Representación elegida:', runs[nombre]['kind'])


### Actividad 7. Interpretar la extensión (opcional)

Comprueba las unidades de los dos cocientes físicos. Si realizaste la comparación, registra los RMSE de validación y explica qué puede aportar esta representación. Si conservaste la ruta básica, desarrolla solamente el análisis de unidades.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 9. Registrar la decisión y evaluar la prueba
La celda siguiente registra la configuración final de cada protocolo. A partir de este punto, usamos esas elecciones para describir el desempeño en prueba. Las nuevas ideas que surjan del análisis pueden registrarse como futuros experimentos con evaluación independiente.

Usaremos tres predictores: media de entrenamiento, referencia lineal y red elegida. El conjunto de prueba es el mismo para los tres dentro de cada protocolo.


In [ ]:
decisiones = pd.DataFrame([
    {'Protocolo': nombre, 'Representación': run['kind'], **run['best_config']}
    for nombre, run in runs.items()])
display(decisiones)

rows = []
for nombre, run in runs.items():
    it, iv, ie = run['parts']
    y = df.iloc[ie].L_dB.to_numpy()
    pred_media = np.full_like(y, df.iloc[it].L_dB.mean())
    pred_lineal = predict_db(run['linear'], run['xs'][2], run['sy'])
    pred_red = predict_db(run['net'], run['xs'][2], run['sy'])
    for modelo, pred in [('Media train', pred_media), ('Lineal', pred_lineal), ('Red', pred_red)]:
        rows.append({'Protocolo': nombre, 'Modelo': modelo, **metrics(y, pred)})
    run['pred_test'] = pred_red
results = pd.DataFrame(rows)
display(results.round(3))


### Actividad 8. Comparar con evidencia

En cada protocolo calcula RMSE(lineal) − RMSE(red) y MAE(lineal) − MAE(red). Explica el signo de cada diferencia. Compara luego el RMSE de la red entre protocolos y relaciona el cambio con sus respectivas preguntas.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 10. Localizar y comprender los errores
Definimos el error firmado como $e_i=\hat L_i-L_i$: un valor positivo indica sobreestimación; uno negativo, subestimación. Su media resume el sesgo global, aunque errores de signos opuestos pueden compensarse.

En el gráfico observado–predicho, la diagonal corresponde al acuerdo perfecto. Los otros paneles muestran errores frente a frecuencia y velocidad. Busca desplazamientos respecto de cero, regiones de mayor dispersión y estructuras persistentes.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for axs, (nombre, run) in zip(axes, runs.items()):
    test = df.iloc[run['parts'][2]]
    pred = run['pred_test']
    error = pred - test.L_dB.to_numpy()
    lo = min(test.L_dB.min(), pred.min())
    hi = max(test.L_dB.max(), pred.max())
    axs[0].scatter(test.L_dB, pred, s=12, alpha=.5)
    axs[0].plot([lo, hi], [lo, hi], 'k--')
    axs[0].set(xlabel='Observado [dB]', ylabel='Predicho [dB]', title=nombre)
    axs[1].scatter(test.f_Hz, error, s=12, alpha=.5)
    axs[1].set_xscale('log')
    axs[1].set(xlabel='Frecuencia [Hz]', ylabel='Error firmado [dB]')
    axs[2].scatter(test.U_ms, error, s=12, alpha=.5)
    axs[2].set(xlabel='Velocidad [m/s]', ylabel='Error firmado [dB]')
    for ax in axs[1:]:
        ax.axhline(0, color='k', linestyle='--')
    print(nombre, 'sesgo medio [dB]:', round(error.mean(), 3))
plt.tight_layout()
plt.show()


### 10.1. Examinar la cobertura de las entradas
La siguiente comparación identifica rangos presentes en entrenamiento, validación y prueba. Los mínimos y máximos resumen la cobertura marginal; la combinación de condiciones de cada fila también influye en la dificultad de predicción.


In [ ]:
for nombre, partes in splits.items():
    print('\n' + nombre)
    rangos = pd.concat({etiqueta: df.iloc[ix][entradas].agg(['min', 'max']).T
                        for etiqueta, ix in zip(['train', 'valid', 'test'], partes)}, axis=1)
    display(rangos)


### Actividad 9. Relacionar errores y física

Describe una región con errores apreciables y apóyate en una figura. Explica qué se puede estudiar en el panel de errores frente a velocidad cuando todo el test tiene una única velocidad. Discute el alcance de la predicción considerando la geometría del perfil, las condiciones medidas y la disponibilidad de δ*.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


## 11. Integrar el análisis y entregar
Completa las respuestas a las actividades obligatorias 1–6 y 8–9. La actividad 7 corresponde a la extensión. Conserva los resultados de las celdas y escribe una discusión final de 500–800 palabras, siguiendo este hilo:

1. Presenta el sistema, las entradas, la salida y tus hipótesis.
2. Explica las particiones y cómo se calcularon las transformaciones.
3. Justifica las configuraciones elegidas usando validación y curvas.
4. Compara las tres referencias en prueba mediante RMSE, MAE y R².
5. Relaciona los errores con las condiciones físicas y formula una conclusión delimitada por la evidencia.

**Criterios de evaluación:** interpretación del sistema y unidades, 20 %; particiones y preprocesamiento, 25 %; comparación de modelos y selección, 25 %; análisis de errores y alcance de las conclusiones, 30 %. Se valora la claridad del procedimiento y la evidencia que sostiene las conclusiones.

**Comprobación de entrega:** reinicia el kernel, ejecuta la ruta elegida en orden y verifica que las figuras y métricas correspondan a esa ejecución. Las semillas ayudan a reproducir el experimento; las versiones de bibliotecas y plataformas pueden introducir pequeñas diferencias.


### Actividad 10. Discusión final

Escribe aquí tu discusión de 500–800 palabras siguiendo el hilo de la sección anterior.


**Tu respuesta:**

Escribe aquí tu análisis. Incluye los valores o figuras que lo sustentan cuando corresponda.


### Fuentes y decisiones de implementación
Datos y descripción física: [UCI Airfoil Self-Noise](https://archive.ics.uci.edu/dataset/291/airfoil+self+noise), DOI [10.24432/C5VW2C](https://doi.org/10.24432/C5VW2C), Brooks, Pope y Marcolini (1989), CC BY 4.0. La copia integrada conserva las seis columnas del archivo de datos.

Implementación: [MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html) y [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). Esta edición usa scikit-learn para facilitar la ejecución en CPU. Las tres configuraciones se comparan sobre una partición fija de validación. El control externo de épocas permite utilizar exactamente la validación definida en cada protocolo.
